In [140]:
import pandas as pd
import datetime
# import pymysql
import pandas.io.sql as psql
from datetime import datetime as dt
import numpy as np
import pandas.tseries.offsets as offsets
import python_ss.python_ss as ps
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import json

import os
import ast
import db_dtypes
from google.cloud import bigquery
from google.oauth2 import service_account
from google.cloud import secretmanager
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
# importでエラーが出てしまった場合は、コマンドプロンプトにて「pip install ”必要なモジュール”」でインストールしていただく必要がございます。
# 例. pip install db_dtypes

# import xlsxwriter

import os
from decimal import Decimal
import calendar
# import utils
# from utils import *
#importlib.reload(utils)
print(os.getcwd())



z:\Users\suehara\Documents\python\analysis\yojitu


In [141]:
pd.options.display.max_rows = 10000
pd.options.display.max_columns = 200


In [142]:
#転機IDの10000以降の手上げ情報取得
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly',
          'https://www.googleapis.com/auth/spreadsheets']
json_path = r"Z:\Users\suehara\Documents\python\analysis\python_ss\credentials.json"
service = ps.get_auth(SCOPES,json_path)
SPREADSHEET_ID = '11scU7ixGvt2JYSBQlHkGMKZSLSzYbrmG221CXUGDqDU'
Sheet_NAME = 'masta!'
Sheet_row = "A:D"
RANGE_NAME = Sheet_NAME+Sheet_row
master1 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)


Sheet_row = "G:M"
RANGE_NAME = Sheet_NAME+Sheet_row
master2 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)


Sheet_row = "O:P"
RANGE_NAME = Sheet_NAME+Sheet_row
master4 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)


Sheet_row = "R:U"
RANGE_NAME = Sheet_NAME+Sheet_row
master5 = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)


In [143]:
master1.dtypes

日付     object
月      object
週      object
計上Q    object
dtype: object

In [144]:
master2.dtypes

Q            object
人マスタ         object
sei_plus     object
user_id      object
職種           object
ロンザン所属フラグ    object
レイヤー         object
dtype: object

In [145]:
master5.dtypes

計上Q         object
修正後ポイント     object
Q計上時ポイント    object
掛け率         object
dtype: object

In [146]:
#ロンザンのマスタデータのパスの設定
path1 = r"\\172.16.0.232\CoffeeCrazy\経営ソリューション事業部\□シニアスカウト事業部□\01 全体進捗\02 行動カレンダー\pythonデータ"

#マスタデータを読み込み
# master1= 日付・月・カレンダー週・Qデータ(2019/10/1	23-10月	9月5W(23日～1日)	23-1Q)
# master1 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[0,1,2,3]).dropna(how='all')

# master2= 在籍Q・人マスタ・略・user_id・所属フラグ・ロンザン所属フラグ(23-2Q	五十嵐奏子	五十嵐　igarashi ミドル	0)
# master2 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[6,7,8,9,10,11])

# master3= 人マスタ・略・チーム・レイヤー(大仲研司	大仲	1課	部責)
# master3 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[14,15,16,17])

# master4= APソースの丸め　ヨミ表選択・丸め(人事部	人事部紹介)
# master4 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[20,21])

# master5= 計上Q・修正後ポイント・Q計上時ポイント・掛け率(19-3Q	5,712	7,297	78%)
# master5 = pd.read_excel(path1 + "\※最新※ロンザン社長資料マスタデータ.xlsx",sheet_name='マスタ',usecols=[23,24,25,26])

In [147]:
#master3 = master3.rename(columns={"人マスタ.1": "人マスタ","sei_plus.1":"sei_plus"}) #カラム名変更
#master3 = master3.dropna(subset=['人マスタ', 'sei_plus'])

master1['日付'] = pd.to_datetime(master1['日付']) 

master2 = master2.dropna(subset=['人マスタ', 'sei_plus'])

master4 = master4.dropna(subset=['ヨミ表選択'])

master5 = master5.rename(columns={"計上Q.1": "計上Q","掛け率.1":"掛け率"}) #カラム名変更
master5 = master5.dropna(subset=['計上Q'])

In [148]:
Sheet_NAME = 'Q営業日!'
Sheet_row = "A:O"
RANGE_NAME = Sheet_NAME+Sheet_row
Q_master = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

Q_master['月'] = pd.to_datetime(Q_master['月'], errors='coerce')

Q_master['日付'] = pd.to_datetime(Q_master['日付'])

In [149]:
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
  access_secret_version('temp-for-sandbox', 'TEMP_CREDENTIAL_KEY'),
  scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
access_secret_version('r-group-bigdata', 'CREDENTIALS_SECRET_KEY_WORKER'),
scopes=["https://www.googleapis.com/auth/cloud-platform"],)



sql = """
SELECT
  CONCAT(period,"-",quarter,"Q") AS Q,
  MIN(date) AS first_date,
  MAX(date) AS end_date
FROM `r-group-bigdata.koyomi.calendar`
GROUP BY CONCAT(period,"-",quarter,"Q")
ORDER BY first_date
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
date_df = client.query(sql).result().to_dataframe()

today = pd.to_datetime(dt.today().strftime('%Y-%m-%d'))

current_quarter_row = date_df[(date_df['first_date'] <= today) & (date_df['end_date'] >= today)]

if not current_quarter_row.empty:
    Q = current_quarter_row.iloc[0]['Q']
    first_date = current_quarter_row.iloc[0]['first_date']
    end_date = current_quarter_row.iloc[0]['end_date']
    print(f"Q: {Q}, First Date: {first_date}, End Date: {end_date}")
else:
    print("本日の日付に該当するクォーターは見つかりませんでした。")

z:\Users\suehara\Documents\python\analysis\.venv\Lib\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Q: 29-4Q, First Date: 2026-07-01, End Date: 2026-10-01


In [150]:
print(first_date)
print(end_date)
print(Q)

2026-07-01
2026-10-01
29-4Q


In [151]:
def access_secret_version(project_id, secret_id, version_id='latest'):
    client = secretmanager.SecretManagerServiceClient()

    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    response = client.access_secret_version(request={"name": name})
    payload = response.payload.data.decode("UTF-8")
    return ast.literal_eval(payload)

# 上記関数を実行するコードが記載されています。こちらもそのままお使いください。
credentials = service_account.Credentials.from_service_account_info(
access_secret_version('r-group-bigdata', 'CREDENTIALS_SECRET_KEY_WORKER'),
scopes=["https://www.googleapis.com/auth/cloud-platform"],)


#社員データ抽出
sql="""
select
user_id ,
sei_plus,
concat(sei,mei) as seimei
FROM `r-group-bigdata.live_company.syain`
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
syain_data = client.query(sql).result().to_dataframe()
syain_data.sample(30)

,user_id,sei_plus,seimei
1829,ku-hayashi,林玖,林玖玲愛
2213,tsubura,螺良早,螺良早彩
3662,kao-kobayashi,小林香,小林香央莉
2456,y-tamashiro,玉城佑,玉城佑馬
416,taguchi,None,田口裕一
438,n-baba,None,馬場奈津子
205,ishioka,石岡諒,石岡諒真
2751,None,None,テストクリプトン
1761,nakazato,中里由,中里由奈
1699,s-hori,堀汐,堀汐里


# 初期交渉

In [152]:
#初期交渉のDBから基本データを抽出（初期交渉データ）
#AP獲得者がいる場合は、AP担当はAP獲得者。
#AP獲得者が空欄でAPソース：パートナー紹介の場合はAP担当は紹介受領者。
#AP獲得者が空欄でAPソース：人事部、転機は面談担当者。
#AP獲得者が空欄でAPソースがパートナー紹介、人事部、転機の場合もAP担当は面談担当者。
#且つAP獲得者がロンザン所属でない場合（所属フラグ空欄）は面談担当者にする。

sql = """
with base3 as (
with base2 as (
with base as (
SELECT
 shoki.id,
 kosho_yoteibi,
 kosho_setteibi,
 mendan_tanto,
 kosho_jisshibi,
 shoki.kohosha_id,
 kohosha_sql.name as kohosha_rank,
 kosho_seq as kaisu,
 case when consts.name = 'HP反響' then 'その他'
       when consts.name = '上場企業役員DM' then 'その他'
       when consts.name = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when consts.name = 'Gアポ' then 'その他'
       else consts.name end as kohosha_apsource,
 shokai_sql.juryosha,
 shoki.ap_kakutoku as shokikosho_apkakutokusha_moto,
 case
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is not null then shokai_sql.juryosha
    when shoki.ap_kakutoku is null and consts.name = "パートナー紹介" and shokai_sql.juryosha is null then shoki.mendan_tanto
    when shoki.ap_kakutoku is null and consts.name != "パートナー紹介" then shoki.mendan_tanto
    else shoki.ap_kakutoku end as shokikosho_apkakutokusha,
   
 case when shoki.partner_id is not null then shoki.partner_id
    when shoki.jinjibu_id is not null then shoki.jinjibu_id
    else null end as juryo_id,

 DATE_DIFF(kosho_setteibi, LAG(kosho_jisshibi) OVER (PARTITION BY shoki.kohosha_id ORDER BY kosho_jisshibi), DAY) AS keikabi
from `r-group-bigdata.live_rhs.shokikoshos` as shoki
left join
 (select id,seimei,kohosha_rank from `r-group-bigdata.live_rhs.kohoshas`) as koho ON shoki.kohosha_id = koho.id
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 5200) as kohosha_sql ON koho.kohosha_rank = kohosha_sql.code
left join
 (select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 19) as consts ON shoki.ap_source = consts.code
left join
 (select kohosha_id,juryosha from `r-group-bigdata.live_rhs.shokaijuryos`) as shokai_sql ON shoki.kohosha_id = shokai_sql.kohosha_id)

select *,
    LAG(kohosha_apsource) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_kohosha_apsource,
  from base)

select *,
   CASE WHEN kohosha_apsource != prev_kohosha_apsource THEN 1 ELSE 0
        END AS APS_change
  from base2)

select
 id,
 kosho_yoteibi,
 kosho_setteibi,
 mendan_tanto,
 kosho_jisshibi,
 kohosha_id,
 kohosha_rank,
 kaisu,
 kohosha_apsource,
 juryosha,
 shokikosho_apkakutokusha_moto,
 shokikosho_apkakutokusha,
 juryo_id,
 keikabi,
 prev_kohosha_apsource,
 APS_change,
 keikabi,
 case when kaisu = 1 then 1
      when APS_change = 1 then 1
      when keikabi > 90 then 2
      else 0 end as sai_flg
from base3
--   where APS_change = 1
--   and kosho_seq >= 4
--   where kohosha_id = 25408
where kosho_setteibi >= date '2017-04-01'

  order by kosho_setteibi desc
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokikosho_data = client.query(sql).result().to_dataframe()

#実施日の古い順に並び替え
shokikosho_data = shokikosho_data.sort_values(by="kosho_jisshibi")

#候補者APソースが空欄の行を削除
shokikosho_data = shokikosho_data.dropna(subset=["kohosha_apsource"])
#重複データを削除
shokikosho_data = shokikosho_data.sort_values(['id','kosho_jisshibi']) 
shokikosho_data = shokikosho_data.drop_duplicates(subset='id')

shokikosho_data["kosho_yoteibi"] = pd.to_datetime(shokikosho_data["kosho_yoteibi"]) #日付データを変換
shokikosho_data["kosho_setteibi"] = pd.to_datetime(shokikosho_data["kosho_setteibi"]) #日付データを変換
shokikosho_data["kosho_jisshibi"] = pd.to_datetime(shokikosho_data["kosho_jisshibi"]) #日付データを変換

#shokikosho_data.to_excel('初期交渉元データ.xlsx',sheet_name='new_sheet_name')

AP獲得者がロンザン外メンバ－時には、AP獲得者を面談担当者に置き換える

In [153]:
master1.dtypes

日付     datetime64[ns]
月              object
週              object
計上Q            object
dtype: object

In [154]:
df = shokikosho_data.copy()

#kosho_setteibi　をマスタデータに紐づけ
df = df.rename(columns={"kosho_setteibi":"日付"}) #カラム名変換
df = pd.merge(df,master1,on = ("日付"),how = "left") #kosho_setteibi関連の日付データを紐付け
df = df.drop(["月"], axis=1).rename(columns={"計上Q":"Q"})

In [155]:
#AP獲得者がロンザン所属外メンバーだった場合、AP獲得者を面談担当者に丸める（0初期交渉AP、1初期交渉実施とならないため）
df = df.rename(columns={"shokikosho_apkakutokusha": "user_id"}) #カラム名変更
df = pd.merge(df,syain_data,on = "user_id",how="left") #名前が変換される
df = pd.merge(df,master2,on = ("sei_plus","Q","user_id"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）




In [156]:
#AP担当がロンザン所属以外は面談担当者を設定する
df["ロンザン所属フラグ"]=df["ロンザン所属フラグ"].fillna("0") #AP担当が空欄（ロンザン所属以外）のデータを埋める
df["AP担当"] = df.apply(lambda x : x["user_id"] if x["ロンザン所属フラグ"] == '1' else x["mendan_tanto"] ,axis = 1) #AP担当がロンザン所属以外は面談担当者を設定する


In [157]:

#必要なカラム名だけに絞る（初期交渉時面談担当がロンザン所属メンバーだったか
df = df[["id","AP担当","sei_plus"]]
#df = df[["kohosha_id","AP担当","user_id","面談担当者","Q","ロンザン所属フラグ"]]
#AP獲得者データを紐付け
shokikosho_data = pd.merge(shokikosho_data,df,on = "id",how="left") 
shokikosho_data = shokikosho_data.drop(["shokikosho_apkakutokusha"], axis=1)



In [158]:
shokikosho_data.sample(10)

,id,kosho_yoteibi,kosho_setteibi,mendan_tanto,kosho_jisshibi,kohosha_id,kohosha_rank,kaisu,kohosha_apsource,juryosha,shokikosho_apkakutokusha_moto,juryo_id,keikabi,prev_kohosha_apsource,APS_change,keikabi_1,sai_flg,AP担当,sei_plus
3342,4649,2018-03-06,2018-03-05,y-ooya,2018-03-06,6370,通常,1,その他,None,None,<NA>,<NA>,None,0,<NA>,1,y-ooya,大矢裕
33287,34758,2021-10-12,2021-09-17,hiraoka,2021-10-12,32407,通常,1,パートナー紹介,hiraoka,hiraoka,15684,<NA>,None,0,<NA>,1,hiraoka,平岡範
73156,74725,2025-02-07,2025-01-29,ichihara,2025-02-07,62255,通常,1,転機社長名鑑,None,naito,<NA>,<NA>,None,0,<NA>,1,ichihara,内藤ち
65458,67027,2024-07-26,2024-07-25,sone,2024-07-26,56317,通常,1,SMAP,None,urasaki,<NA>,<NA>,None,0,<NA>,1,sone,浦崎紋
34991,36462,2021-11-22,2021-11-17,nagase,2021-11-22,33610,通常,1,顧問名鑑登録 解放者,None,nagase,<NA>,<NA>,None,0,<NA>,1,nagase,永瀬絢
45100,46580,2022-12-08,2022-12-05,yoshitake,2022-12-08,40253,通常,1,SMAP,None,s-kobayashi,<NA>,<NA>,None,0,<NA>,1,s-kobayashi,小林清
29040,30492,2021-06-10,2021-06-09,kitazawa,2021-06-10,29681,通常,1,転機社長名鑑,None,naito,<NA>,<NA>,None,0,<NA>,1,kitazawa,内藤ち
38579,40054,2022-05-09,2022-04-27,d-katsumata,2022-05-09,36033,通常,1,転機社長名鑑,None,naito,<NA>,<NA>,None,0,<NA>,1,d-katsumata,内藤ち
60216,61782,2024-03-19,2024-03-16,iwayoshi,2024-03-19,4350,通常,2,SMAP,None,otsuka,<NA>,2251,転機社長名鑑,1,2251,1,otsuka,大塚洋
4630,5954,2018-06-11,2018-06-11,nagamatsu,2018-06-11,7762,通常,1,顧問名鑑登録 解放者,None,None,<NA>,<NA>,None,0,<NA>,1,nagamatsu,長松大


候補者がロンザン候補者か否かを初期交渉時の候補者担当の所属で判断する


### 初期交渉データを整える

In [159]:
#候補者APソース　追加
shokikosho_data = shokikosho_data.rename(columns={"kohosha_apsource":"ヨミ表選択"}) #カラム名変換
shokikosho_data = pd.merge(shokikosho_data,master4,on = ("ヨミ表選択"),how = "left") #候補者APソースを紐付け
shokikosho_data = shokikosho_data.rename(columns={"丸め":"APソース丸め"})

#初期交渉設定週（カレンダー用）　追加
shokikosho_data = shokikosho_data.rename(columns={"kosho_setteibi":"日付"}) #カラム名変換
shokikosho_data = pd.merge(shokikosho_data,master1,on = ("日付"),how = "left") #kosho_setteibi関連の日付データを紐付け
shokikosho_data = shokikosho_data.rename(columns={"週":"初期交渉設定週","日付":"kosho_setteibi"})

#初期交渉実施週（カレンダー用）　追加
shokikosho_data = shokikosho_data.rename(columns={"kosho_jisshibi":"日付"}) #カラム名変換
shokikosho_data = pd.merge(shokikosho_data,master1,on = ("日付"),how = "left") #kosho_setteibi関連の日付データを紐付け
shokikosho_data = shokikosho_data.rename(columns={"週":"初期交渉実施週","日付":"kosho_jisshibi"})

# shokikosho_data.to_excel('shokikosho_data.xlsx',sheet_name='new_sheet_name')

In [160]:
#初期交渉設定だけのテーブル
shokikosho_settei = shokikosho_data[['id','kosho_setteibi','AP担当','sei_plus','APソース丸め','sai_flg','kaisu','初期交渉設定週']].copy()
shokikosho_settei = shokikosho_settei.rename(columns={"kosho_setteibi":"日付","AP担当":"担当者","初期交渉設定週":"ｶﾚﾝﾀﾞｰ週"})
shokikosho_settei["value"] = shokikosho_settei.apply(lambda x : 1 if x["sai_flg"] == 1 else 0 ,axis = 1)
shokikosho_settei['type'] = 'shoki_ap'
shokikosho_settei['組手'] = ''

# shokikosho_settei.to_excel('shokikosho_settei.xlsx',sheet_name='new_sheet_name')

In [161]:
shokikosho_data

,id,kosho_yoteibi,kosho_setteibi,mendan_tanto,kosho_jisshibi,kohosha_id,kohosha_rank,kaisu,ヨミ表選択,juryosha,shokikosho_apkakutokusha_moto,juryo_id,keikabi,prev_kohosha_apsource,APS_change,keikabi_1,sai_flg,AP担当,sei_plus,APソース丸め,月_x,初期交渉設定週,計上Q_x,月_y,初期交渉実施週,計上Q_y
0,807,2017-04-05,2017-04-03,oonaka,2017-04-05,2033,通常,1,パートナー紹介,oonaka,None,1008,<NA>,None,0,<NA>,1,oonaka,大仲研,パートナー紹介,20-4月,,20-3Q,20-4月,,20-3Q
1,808,2017-04-19,2017-04-03,oonaka,2017-04-19,2034,準VIP,1,人事部経由,oonaka,None,136,<NA>,None,0,<NA>,1,oonaka,大仲研,人事部紹介,20-4月,,20-3Q,20-4月,,20-3Q
2,809,2017-04-05,2017-04-03,oonaka,2017-04-05,2035,準VIP,1,パートナー紹介,oonaka,None,587,<NA>,None,0,<NA>,1,oonaka,大仲研,パートナー紹介,20-4月,,20-3Q,20-4月,,20-3Q
3,811,2017-04-11,2017-04-04,y-shirai,2017-04-11,2039,準VIP,1,パートナー紹介,y-shirai,None,949,<NA>,None,0,<NA>,1,y-shirai,白井友,パートナー紹介,20-4月,,20-3Q,20-4月,,20-3Q
4,812,2017-04-07,2017-04-04,oonaka,2017-04-07,2022,準VIP,1,人事部経由,oonaka,None,108,<NA>,None,0,<NA>,1,oonaka,大仲研,人事部紹介,20-4月,,20-3Q,20-4月,,20-3Q
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92614,94195,2026-07-31,2026-07-30,no-uchida,NaT,75448,通常,2,転機社長名鑑,None,naito,<NA>,<NA>,SMAP,1,<NA>,1,no-uchida,内藤ち,転機,29-7月,7月5W(27日-31日),29-4Q,NaN,NaN,NaN
92615,94196,2026-08-06,2026-07-30,ry-maeda,NaT,77531,通常,1,パートナー紹介,ry-maeda,ry-maeda,26522,<NA>,None,0,<NA>,1,ry-maeda,前田崚,パートナー紹介,29-7月,7月5W(27日-31日),29-4Q,NaN,NaN,NaN
92616,94197,2026-08-04,2026-07-30,r-yoshimura,NaT,41298,通常,3,その他,None,r-yoshimura,<NA>,<NA>,顧問名鑑登録 解放者,1,<NA>,1,r-yoshimura,吉村理,その他,29-7月,7月5W(27日-31日),29-4Q,NaN,NaN,NaN
92617,94198,2026-07-30,2026-07-30,sho-saito,2026-07-30,76907,準VIP,2,SMAP,None,sho-saito,<NA>,21,SMAP,0,21,0,sho-saito,齋藤勝,SMAP,29-7月,7月5W(27日-31日),29-4Q,29-7月,7月5W(27日-31日),29-4Q


In [162]:


#初期交渉実施だけのテーブル(交渉実施日がない案件は未実施のため行削除)
shokikosho_jisshi = shokikosho_data.copy().drop(['sei_plus'],axis=1)
shokikosho_jisshi = shokikosho_jisshi.rename(columns={"mendan_tanto":"user_id"})
shokikosho_jisshi = pd.merge(shokikosho_jisshi,syain_data,on = "user_id",how="left") #名前が変換される


shokikosho_jisshi = shokikosho_jisshi[['id','kosho_jisshibi','user_id','sei_plus','APソース丸め','sai_flg','kaisu','初期交渉実施週']]
shokikosho_jisshi = shokikosho_jisshi.rename(columns={"kosho_jisshibi":"日付","user_id":"担当者","初期交渉実施週":"ｶﾚﾝﾀﾞｰ週"}).dropna(subset=['日付'])
shokikosho_jisshi["value"] = shokikosho_jisshi.apply(lambda x : 1 if x["sai_flg"] == 1 else 0 ,axis = 1)
shokikosho_jisshi['type'] = 'shoki_jissi'
shokikosho_jisshi['組手'] = ''



In [163]:
shokikosho_settei.columns

Index(['id', '日付', '担当者', 'sei_plus', 'APソース丸め', 'sai_flg', 'kaisu', 'ｶﾚﾝﾀﾞｰ週',
       'value', 'type', '組手'],
      dtype='object')

In [164]:
shokikosho_jisshi.columns

Index(['id', '日付', '担当者', 'sei_plus', 'APソース丸め', 'sai_flg', 'kaisu', 'ｶﾚﾝﾀﾞｰ週',
       'value', 'type', '組手'],
      dtype='object')

In [165]:
#初期交渉設定と実施をそれぞれ同じ形で整える
shokikosho = pd.concat([shokikosho_settei,shokikosho_jisshi],ignore_index=True)
#Q情報　追加
shokikosho = pd.merge(shokikosho,Q_master,on = ("日付"),how = "left")

# shokikosho.to_excel('shokikosho.xlsx',sheet_name='new_sheet_name')

# 本交渉設定

In [166]:
sql = """
WITH
-- 1. 初期交渉のベースデータと必要な計算（不要なJOINを削除しスッキリさせました）
shoki_base AS (
  SELECT
    shk.id,
    shk.kohosha_id,
    shk.ap_source,
    -- COALESCE関数を使って、nullの場合の処理を1行で記述します
    COALESCE(syi1.sei_plus, shk.mendan_tanto) AS mendan_tanto,
    syi2.sei_plus AS ap_kakutokusha,
    shk.kosho_setteibi,
    shk.kosho_jisshibi,
    shk.kosho_seq,
    -- 前回実績日からの経過日数を計算
    DATE_DIFF(shk.kosho_setteibi, LAG(shk.kosho_jisshibi) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_jisshibi), DAY) AS keikabi,
    -- 前回のAPソースを取得
    LAG(shk.ap_source) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_setteibi) AS prev_APsource
  FROM `r-group-bigdata.live_rhs.shokikoshos` AS shk
  LEFT JOIN `r-group-bigdata.live_company.syain` AS syi1 
    ON shk.mendan_tanto = syi1.user_id
  LEFT JOIN `r-group-bigdata.live_company.syain` AS syi2 
    ON shk.ap_kakutoku = syi2.user_id
),

-- 2. 初期交渉のフラグ計算（複数あったステップを1つにまとめました）
shoki_flags AS (
  SELECT
    id,
    kosho_jisshibi,
    kosho_seq,
    ap_source,
    CASE
      WHEN kosho_seq = 1 THEN 1
      WHEN ap_source != prev_APsource THEN 1
      WHEN keikabi > 90 THEN 2
      ELSE 0
    END AS sai_flg
  FROM shoki_base
),

-- 3. 本交渉の対象データ（★ここで先に条件を絞り込むことで、処理速度が大幅にアップします）
target_hons AS (
  SELECT
    id,
    anken_id,
    kosho_setteibi,
    kohosha_tanto,
    kosho_seq,
    kosho_seq_extra,
    kosho_jisshibi,
    mendan_tanto
  FROM `r-group-bigdata.live_rhs.honkoshos`
  WHERE kosho_setteibi >= '2017-04-01'
    AND kosho_seq = 1
)

-- 4. 最終的な結合と出力（サブクエリを無くし、ON句で条件を指定して読みやすくしました）
SELECT
  hon.id,
  hon.anken_id,
  hon.kosho_setteibi,
  an.kohosha_id,
  hon.kohosha_tanto,
  an.kigyo_tanto,
  kohosha_ap_sql.name AS moto_apsource,
  COALESCE(kohosha_ap_sql2.name, kohosha_ap_sql.name) AS kohosha_apsource,
  kohosha_sql.name AS kohosha_rank,
  kgy.name AS company_name,
  hon.kosho_seq AS kaisu,
  hon.kosho_seq_extra AS absolute_1,
  hon.kosho_jisshibi,
  hon.mendan_tanto,
  honkosho_kumite_sql.name AS kumite_moto,
  FORMAT_DATE('%Y/%m/%d', shoki.kosho_jisshibi) AS shoki_jisshibi,
  shoki.kosho_seq,
  shoki.sai_flg,
  DATE_DIFF(hon.kosho_setteibi, shoki.kosho_jisshibi, DAY) AS jisshibi_sa,
  -- 元のクエリの最後のSELECT文で行っていた計算をここに統合しました
  CASE
    WHEN shoki.kosho_jisshibi IS NULL THEN 2
    WHEN DATE_DIFF(hon.kosho_setteibi, shoki.kosho_jisshibi, DAY) > 90 THEN 2
    ELSE shoki.sai_flg
  END AS sai_flg2,
  case when an.hanjokin = 10 then "通常"
       when an.hanjokin = 20 then "半常勤"
       else "-" end as hanjokin
FROM target_hons AS hon
LEFT JOIN `r-group-bigdata.live_rhs.ankens` AS an
  ON hon.anken_id = an.id
LEFT JOIN shoki_flags AS shoki
  ON an.linked_shokikosho_id = shoki.id
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` AS koho
  ON an.kohosha_id = koho.id
LEFT JOIN `r-group-bigdata.live_rhs.kigyos` AS kgy
  ON an.kigyo_id = kgy.id
LEFT JOIN `r-group-bigdata.live_rhs.sys_consts` AS kohosha_sql
  ON koho.kohosha_rank = kohosha_sql.code AND kohosha_sql.group_code = 5200
LEFT JOIN `r-group-bigdata.live_rhs.sys_consts` AS kohosha_ap_sql
  ON koho.ap_source = kohosha_ap_sql.code AND kohosha_ap_sql.group_code = 19
LEFT JOIN `r-group-bigdata.live_rhs.sys_consts` AS kohosha_ap_sql2
  ON shoki.ap_source = kohosha_ap_sql2.code AND kohosha_ap_sql2.group_code = 19
LEFT JOIN `r-group-bigdata.live_rhs.sys_consts` AS honkosho_kumite_sql
  ON an.kumite = honkosho_kumite_sql.code AND honkosho_kumite_sql.group_code = 4500
;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honkosho_data = client.query(sql).result().to_dataframe()

#候補者APソースが空欄の行を削除
honkosho_data = honkosho_data.dropna(subset=["kohosha_apsource"])

honkosho_data["kosho_setteibi"] = pd.to_datetime(honkosho_data["kosho_setteibi"]) #日付データを変換
honkosho_data["kosho_jisshibi"] = pd.to_datetime(honkosho_data["kosho_jisshibi"]) #日付データを変換

#候補者担当が"usuda","ikke","akamatsu"のデータを除外
honkosho_data = honkosho_data[~honkosho_data["kohosha_tanto"].isin(["usuda","ikke","akamatsu"])]

In [167]:
honkosho_data

,id,anken_id,kosho_setteibi,kohosha_id,kohosha_tanto,kigyo_tanto,moto_apsource,kohosha_apsource,kohosha_rank,company_name,kaisu,absolute_1,kosho_jisshibi,mendan_tanto,kumite_moto,shoki_jisshibi,kosho_seq,sai_flg,jisshibi_sa,sai_flg2,hanjokin
0,18196,17314,2023-10-13,36622,hasui,kakuta,SMAP,SMAP,通常,ネクストエナジー・アンド・リソース（株）,1,<NA>,2023-10-17,kakuta,両手,2023/10/10,5,2,3,2,-
1,15000,14218,2022-07-22,36622,otsuka,honjo,SMAP,SMAP,通常,ホームテック（株）,1,<NA>,2022-08-26,honjo,片手,2022/06/07,1,1,45,1,-
2,15230,14436,2022-08-21,36622,otsuka,m-yoneda,SMAP,SMAP,通常,ＳＳＦホールディングス（株）,1,<NA>,2022-09-02,m-yoneda,片手,2022/06/07,1,1,75,1,-
3,16116,15296,2023-01-05,36622,sakamaki,sasahara,SMAP,SMAP,通常,（株）カシワバラ・コーポレーション,1,<NA>,NaT,sasahara,両手,None,<NA>,<NA>,<NA>,2,-
4,25815,24605,2025-09-24,69029,i-kitayama,yu-kaneko,転機社長名鑑,転機社長名鑑,通常,（株）ＡＱ Ｇｒｏｕｐ,1,<NA>,2025-10-03,yu-kaneko,両手,2025/09/03,1,1,21,1,通常
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26689,2330,2279,2018-01-24,4683,t-masuda,yamakoshi,パートナー紹介,パートナー紹介,通常,（株）プレジィール,1,<NA>,2018-01-29,yamakoshi,None,2017/11/17,1,1,68,1,-
26690,2182,2146,2017-12-17,4683,t-masuda,t-koga,パートナー紹介,パートナー紹介,通常,小柳建設（株）,1,<NA>,NaT,t-koga,None,2017/11/17,1,1,30,1,-
26691,2105,2072,2017-11-30,4683,t-masuda,morinaga,パートナー紹介,パートナー紹介,通常,万田発酵（株）,1,<NA>,2018-01-15,morinaga,None,2017/11/17,1,1,13,1,-
26692,6939,6479,2019-11-14,13739,yuk-ono,k-takeda,転機社長名鑑,転機社長名鑑,通常,しろくま電力（株）,1,<NA>,2019-11-24,k-takeda,None,2019/07/30,1,1,107,2,-


#再交渉フラグのデータを読み込む
sql = """
select
Honkosho_id as id,
saikosho_flag,
case when saikosho_flag = 1 then "再交渉"
else "-" end as sai_flg
from
`temp-for-sandbox.ronzanmi__mart.vw_honkosho_saikoshos`
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
saikosho_data = client.query(sql).result().to_dataframe()

#本交渉データと再交渉フラグのデータを紐付ける
honkosho_data = pd.merge(honkosho_data,saikosho_data,on = "id",how="left")
#再交渉アポソースを設定する
honkosho_data["saikosho_apsource"] = honkosho_data.apply(lambda x : "再交渉" if x["sai_flg"] == "再交渉" else x["kohosha_apsource"] ,axis = 1)
#重複データを削除
honkosho_data = honkosho_data.drop_duplicates(subset='id')

本交渉データに　社数別・候補者別・組手情報追加

In [168]:
#========================================
#組手用
#========================================
df = honkosho_data.copy()
#本交渉設定日　をマスタデータに紐づけ
df = df.rename(columns={"kosho_setteibi":"日付"}) #カラム名変換
df = pd.merge(df,master1,on = ("日付"),how = "left") #kosho_setteibi関連の日付データを紐付け
df = df.drop(["月"], axis=1).rename(columns={"計上Q":"Q"})


In [169]:
df.columns
syain_data.columns

Index(['user_id', 'sei_plus', 'seimei'], dtype='object')

In [170]:

#候補者担当がロンザン所属外メンバーかどうか確認
df_kohosha = df.rename(columns={"kohosha_tanto": "user_id"}).copy() #カラム名変更
df_kohosha = pd.merge(df_kohosha,syain_data,on = "user_id",how="left").drop("seimei",axis=1) #名前が変換される
df_kohosha["sei_plus"] = df_kohosha["sei_plus"].fillna(df_kohosha["user_id"])
df_kohosha = pd.merge(df_kohosha,master2,on = ("sei_plus","Q","user_id"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）
df_kohosha["ロンザン所属フラグ"] = df_kohosha["ロンザン所属フラグ"].fillna("0") #AP担当が空欄（ロンザン所属以外）のデータを埋める

In [171]:

df_kohosha = df_kohosha.rename(columns={"ロンザン所属フラグ":"候_ロンザン所属フラグ","user_id":"kohosha_tanto","sei_plus":"候_略氏名"})
df_kohosha = df_kohosha[["id","候_ロンザン所属フラグ","kumite_moto","候_略氏名"]]#必要項目のみにする



In [172]:
df_kohosha.to_excel('df_kohosha.xlsx',sheet_name='new_sheet_name')

In [173]:
#企業担当がロンザン所属外メンバーかどうか確認
df_kigyo = df.rename(columns={"kigyo_tanto": "user_id"}).copy() #カラム名変更
df_kigyo = pd.merge(df_kigyo,syain_data,on = "user_id",how="left").drop("seimei",axis=1) #名前が変換される
df_kigyo = pd.merge(df_kigyo,master2,on = ("sei_plus","Q","user_id"),how = "left") #名前が変換される（変換されない人は、そのQにロンザンじゃなかった）
df_kigyo["ロンザン所属フラグ"] = df_kigyo["ロンザン所属フラグ"].fillna("0") #AP担当が空欄（ロンザン所属以外）のデータを埋める
df_kigyo = df_kigyo.rename(columns={"ロンザン所属フラグ":"企_ロンザン所属フラグ","user_id":"kigyo_tanto","sei_plus":"企_略氏名"})
df_kigyo = df_kigyo[["id","企_ロンザン所属フラグ","企_略氏名"]]#必要項目のみにする

df2 = pd.merge(df_kohosha,df_kigyo,on = "id",how="left") #名前が変換される

#企業担当 も 候：ロンザン×企：ロンザン=両手、 残りのケースは片手
df2["候_ロンザン所属フラグ"] = df2["候_ロンザン所属フラグ"].astype(np.int64)
df2["企_ロンザン所属フラグ"] = df2["企_ロンザン所属フラグ"].astype(np.int64)
df2["両手フラグ"] = df2["候_ロンザン所属フラグ"] * df2["企_ロンザン所属フラグ"]
df2["両手フラグ"] = df2["両手フラグ"].apply(lambda x : "両手" if x == 1 else "片手") #候補者担当、企業担当どちらもロンザンの場合は両手、それ以外は片手にする

#本交渉データの「組手」情報があればそちらを採用、本交渉データ上の「組手」がnullのときはロンザン担当者かどうかで判断した「両手フラグ」に置き換える
df2["組手"] = df2.apply(lambda x : x["kumite_moto"] if x["kumite_moto"] is not None else x["両手フラグ"] ,axis = 1)
df2 = df2.drop("kumite_moto",axis=1)
#組手情報を紐付け
honkosho_data = pd.merge(honkosho_data,df2,on = "id",how="left") 
#df2.to_excel('df2.xlsx',sheet_name='new_sheet_name')
del df2

## 本交渉（人）

In [174]:
sql = """
with HON_NINS2 as (
with HON_NINS as (
with SHOKIS as (
  with shoki_base3 as (
with shoki_base2 as (
with shoki_base as (
SELECT
  shk.id,
  shk.tenki_id,
  shk.kohosha_id,
  shk.ap_source,
  shk.annual_income,
  layer.name as layer,
  CASE WHEN syi1.sei_plus IS NULL THEN shk.mendan_tanto
       ELSE syi1.sei_plus END AS mendan_tanto,
  syi2.sei_plus AS ap_kakutokusha,
  shk.kosho_setteibi,
  shk.kosho_yoteibi,
  shk.kosho_jisshibi,
  shk.kosho_seq,
  shk.saikosho_kaisu,
  saikosho_seq,
  valid_flag,
  shk.initial_contact_date,
  DATE_DIFF(shk.kosho_setteibi, LAG(shk.kosho_jisshibi) OVER (PARTITION BY shk.kohosha_id ORDER BY shk.kosho_jisshibi), DAY) AS keikabi
FROM (
  SELECT *,
    FIRST_VALUE(kosho_jisshibi IGNORE NULLS) OVER (PARTITION BY kohosha_id ORDER BY kosho_jisshibi) AS initial_contact_date
  FROM `r-group-bigdata.live_rhs.shokikoshos`
) shk
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` koho ON shk.kohosha_id = koho.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 ON shk.mendan_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 ON shk.ap_kakutoku = syi2.user_id
LEFT JOIN (SELECT 
            code,
            name
           FROM `r-group-bigdata.live_rhs.sys_consts`
           WHERE group_code = 19) consts ON shk.ap_source = consts.code
left join (SELECT 
            group_code,
             code,
             name
            FROM `r-group-bigdata.live_rhs.sys_consts`
            where group_code = 25) layer on shk.max_bushoyakushoku = layer.code
ORDER BY shk.kosho_setteibi
)

  select *,
    LAG(ap_source) OVER (PARTITION BY kohosha_id ORDER BY kosho_setteibi) AS prev_APsource,
  from shoki_base)

  select *,
   CASE WHEN ap_source != prev_APsource THEN 1 ELSE 0
        END AS APS_change
  from shoki_base2)

  select *,
  case when kosho_seq = 1 then 1
        when APS_change = 1 then 1
        when keikabi > 90 then 2
        else 0 end as sai_flg
  from shoki_base3
  order by kosho_setteibi desc
)

SELECT 
  hon.id as honkosho_id,
  hon.anken_id as anken_id,
  an.kohosha_id as kohosha_id,
  an.linked_shokikosho_id,
  kgy.tsr_code as tsr_code,
  kgy.name as company_name,
  case when consts.name is null then consts2.name
       else consts.name end as ap_source,
  shoki.annual_income,
  shoki.layer,
  initial_contact_date,     
  format_date('%Y/%m/%d',shoki.kosho_setteibi) as shoki_setteibi,
  format_date('%Y/%m/%d',shoki.kosho_jisshibi) as shoki_jisshibi,
  format_date('%Y/%m/%d',hon.kosho_setteibi) as hon_setteibi,
  format_date('%Y/%m/%d',hon.kosho_yoteibi) as hon_yoteibi,
  format_date('%Y/%m/%d',hon.kosho_jisshibi) as hon_jisshibi,
  case when syi1.sei_plus is null then hon.kohosha_tanto
  else syi1.sei_plus end as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  Q.Q,
  shoki.kosho_seq,
  sai_flg,
  case when hon.kosho_yoteibi >= CURRENT_DATE("Asia/Tokyo") then '実施前'
       when hon.jisshi_flag=2 and hon.deleted=2 then "CXL"
       when hon.jisshi_flag=0 and hon.deleted=0 and hon.nittei_chosei=1 then "日程調整中"
       when hon.jisshi_flag=1 then '実施'
       when hon.jisshi_flag=0 then '未報告'
       else cast(hon.jisshi_flag as string) end as status
FROM `r-group-bigdata.live_rhs.honkoshos` hon
LEFT JOIN `r-group-bigdata.live_rhs.ankens` an on hon.anken_id = an.id
LEFT JOIN SHOKIS shoki on an.linked_shokikosho_id = shoki.id
LEFT JOIN `r-group-bigdata.live_rhs.kigyos` kgy on an.kigyo_id = kgy.id
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` koho on an.kohosha_id = koho.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 on hon.kohosha_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
left join (SELECT yyyymmdd,
           concat(ki,"-",q,"Q") Q
           FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
           order by yyyymmdd) Q on hon.kosho_setteibi = Q.yyyymmdd
LEFT JOIN (SELECT code,name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 19) consts on shoki.ap_source = consts.code
LEFT JOIN (SELECT code,name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 19) consts2 on koho.ap_source = consts2.code
WHERE hon.kosho_seq = 1)

select *,
   case when ap_source = '転機社長名鑑' then '転機'
       when ap_source = '人事部経由' then '人事部紹介'
       when ap_source = '顧問名鑑登録　解放者' then '顧問名鑑登録者' 
       when ap_source = 'HP反響' then 'その他'
       when ap_source = '上場企業役員DM' then 'その他'
       when ap_source = '社外取締役名鑑　候補者' then '顧問名鑑登録　解放者'
       when ap_source = 'Gアポ' then 'その他'
       else ap_source end as ap_source,
 DATE_DIFF(PARSE_DATE('%Y/%m/%d', hon_setteibi), PARSE_DATE('%Y/%m/%d', shoki_jisshibi), DAY) as jisshibi_sa,
 ROW_NUMBER() OVER (PARTITION BY kohosha_id, Q ORDER BY hon_setteibi asc) as rn
from HON_NINS
order by hon_setteibi)

select *,
 case when shoki_jisshibi is null then 2
       when jisshibi_sa > 90 then 2
       else sai_flg end as sai_flg2 
from HON_NINS2
where rn = 1

"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honnin_data = client.query(sql).result().to_dataframe()

In [175]:
#========================================
#候補者別用
#========================================
df = honnin_data.copy()
df3 = df[["honkosho_id","kohosha_id","Q"]].copy()
# df3 = df3.loc[df3["kaisu"] == 1]

#候補者(人)別カウント用フラグ
df3["kohosha_id"] = df3["kohosha_id"].astype(str)
df3["候補者重複用"] = df3["kohosha_id"] + "★" + df3["Q"]

#重複データを表示
df3 = df3.sort_values(['候補者重複用','honkosho_id'])

#重複営業にカウントしたいので、重複用&本交渉idカラムにてまずは並び替え,
df3['候補者重複'] = df3['候補者重複用'].groupby((df3['候補者重複用'] != df3['候補者重複用'].shift()).cumsum()).cumcount() + 1

df3 = df3.rename(columns={"honkosho_id":"id"})

df3 = df3[["id","候補者重複用","候補者重複"]]
# df3["id"] = df3["id"].astype(str).str.replace('<NA>', '')
# df3["id"] = df3["id"].astype('int64')



In [176]:
#候補者カウント情報を紐付け
# df3["id"] = df3["id"].astype(str)


In [177]:
honkosho_data = pd.merge(honkosho_data,df3,on = "id",how="left")
del df3


In [178]:
sql = """
with HON_SHA as (
with base as (
SELECT 
  hon.id as honkosho_id,
  hon.anken_id as anken_id,
  an.kohosha_id as kohosha_id,
  an.linked_shokikosho_id,
  kgy.tsr_code as tsr_code,
  kgy.name as company_name,
  consts.name as ap_source,
  format_date('%Y/%m/%d',shoki.kosho_setteibi) as shoki_setteibi,
  format_date('%Y/%m/%d',shoki.kosho_jisshibi) as shoki_jisshibi,
  format_date('%Y/%m/%d',hon.kosho_setteibi) as hon_setteibi,
  format_date('%Y/%m/%d',hon.kosho_yoteibi) as hon_yoteibi,
  format_date('%Y/%m/%d',hon.kosho_jisshibi) as hon_jisshibi,
  case when syi1.sei_plus is null then hon.kohosha_tanto
  else syi1.sei_plus end as kohosha_tanto,
  syi2.sei_plus as kigyo_tanto,
  Q.Q
FROM `r-group-bigdata.live_rhs.honkoshos` hon
LEFT JOIN `r-group-bigdata.live_rhs.ankens` an on hon.anken_id = an.id
LEFT JOIN `r-group-bigdata.live_rhs.shokikoshos` shoki on an.linked_shokikosho_id = shoki.id
LEFT JOIN `r-group-bigdata.live_rhs.kigyos` kgy on an.kigyo_id = kgy.id
LEFT JOIN `r-group-bigdata.live_rhs.kohoshas` khs on an.kohosha_id = khs.id
LEFT JOIN `r-group-bigdata.live_company.syain` syi1 on hon.kohosha_tanto = syi1.user_id
LEFT JOIN `r-group-bigdata.live_company.syain` syi2 on an.kigyo_tanto = syi2.user_id
left join (SELECT yyyymmdd,
           concat(ki,"-",q,"Q") Q
           FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`
           order by yyyymmdd) Q on hon.kosho_setteibi = Q.yyyymmdd
LEFT JOIN (SELECT code,name
             FROM `r-group-bigdata.live_rhs.sys_consts`
             where group_code = 19) consts on shoki.ap_source = consts.code
where hon.kosho_seq = 1)

select 
*,
 ROW_NUMBER() OVER (PARTITION BY tsr_code, Q ORDER BY hon_setteibi asc) as rn
from base
order by hon_setteibi)

select *
from HON_SHA
where rn = 1
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
honsha_data = client.query(sql).result().to_dataframe()

In [179]:
#========================================
#社数別用
#========================================
df = honsha_data.copy()
df4 = df[["Q","kohosha_id","honkosho_id","company_name"]].copy()
# df4 = df4.loc[df4["kaisu"] == 1]

#社数別カウント用フラグ
df4["社数カウント用"] = df4["Q"] + "★" + df4["company_name"]

#重複データを表示
df4 = df4.sort_values(['社数カウント用','honkosho_id'])

   #重複営業にカウントしたいので、重複用&本交渉idカラムにてまずは並び替え
df4['社数カウント'] = df4['社数カウント用'].groupby((df4['社数カウント用'] != df4['社数カウント用'].shift()).cumsum()).cumcount() + 1
df4 = df4[['honkosho_id','社数カウント用','社数カウント']]


In [180]:
df4 = df4.rename(columns={"honkosho_id":"id"})

In [181]:
# df4["id"] = df4["id"].astype(str)

In [182]:
#社数カウント情報を紐付け

honkosho_data = pd.merge(honkosho_data,df4,on = "id",how="left") 
del df4



### 本交渉データを整える

In [183]:
#候補者APソース　追加
honkosho_data = honkosho_data.rename(columns={"kohosha_apsource":"ヨミ表選択"}) #カラム名変換
honkosho_data = pd.merge(honkosho_data,master4,on = ("ヨミ表選択"),how = "left") #候補者APソースを紐付け
honkosho_data = honkosho_data.rename(columns={"丸め":"APソース丸め"})
#本交渉設定週（カレンダー用）　追加
honkosho_data = honkosho_data.rename(columns={"kosho_setteibi":"日付"}) #カラム名変換
honkosho_data = pd.merge(honkosho_data,master1,on = ("日付"),how = "left") #本交渉設定日関連の日付データを紐付け
honkosho_data = honkosho_data.rename(columns={"週":"本交渉設定週","日付":"kosho_setteibi"})
#本交渉実施週（カレンダー用）　追加
honkosho_data = honkosho_data.rename(columns={"kosho_jisshibi":"日付"}) #カラム名変換
honkosho_data = pd.merge(honkosho_data,master1,on = ("日付"),how = "left") #本交渉設定日関連の日付データを紐付け
honkosho_data = honkosho_data.rename(columns={"週":"本交渉実施週","日付":"kosho_jisshibi"})

In [184]:
# honkosho_data.to_excel('honkosho-raw.xlsx',sheet_name='new_sheet_name')


In [185]:
#候補者担当側
#本交渉設定だけのテーブル
honkosho_settei = honkosho_data[['id','kosho_setteibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉設定週']].copy()
honkosho_settei = honkosho_settei.rename(columns={"kosho_setteibi":"日付","kohosha_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'})
# honkosho_settei["value"] = honkosho_settei.apply(lambda x : 1 if x["sai_flg2"]  in [0, 1, 2] else 0 ,axis = 1)
# 以下がダメだったとき、消す
honkosho_settei["value"] = honkosho_settei.apply(lambda x : 1 if (x["sai_flg2"] in [0, 1, 2]) and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_settei['type'] = 'settei'
# 以下がダメだったとき、消す
# honkosho_settei = honkosho_settei[['id','日付','担当者','sei_plus','APソース丸め','sai_flg2','kaisu','組手','ｶﾚﾝﾀﾞｰ週','value','type']]

honkosho_settei_kohosha = honkosho_data[['id','kosho_setteibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉設定週','候補者重複']].copy()
honkosho_settei_kohosha = honkosho_settei_kohosha.loc[honkosho_settei_kohosha["候補者重複"] == 1]
honkosho_settei_kohosha = honkosho_settei_kohosha.rename(columns={"kosho_setteibi":"日付","kohosha_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).drop(["候補者重複"], axis=1)
honkosho_settei_kohosha["value"] = honkosho_settei_kohosha.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2])  and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_settei_kohosha['type'] = 'settei_kohosha'

honkosho_settei_shinki = honkosho_data[['id','kosho_setteibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉設定週']].copy()
honkosho_settei_shinki = honkosho_settei_shinki.rename(columns={"kosho_setteibi":"日付","kohosha_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'})
honkosho_settei_shinki["value"] = honkosho_settei_shinki.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1])  and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_settei_shinki['type'] = 'settei_shinki'

honkosho_settei_sai = honkosho_data[['id','kosho_setteibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉設定週']].copy()
honkosho_settei_sai = honkosho_settei_sai.rename(columns={"kosho_setteibi":"日付","kohosha_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'})
honkosho_settei_sai["value"] = honkosho_settei_sai.apply(lambda x : 1 if (x["sai_flg2"]  ==2)  and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_settei_sai['type'] = 'settei_sai'

honkosho_settei_company = honkosho_data[['id','kosho_setteibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉設定週','社数カウント']].copy()
honkosho_settei_company = honkosho_settei_company.loc[honkosho_settei_company["社数カウント"] == 1]
honkosho_settei_company = honkosho_settei_company.rename(columns={"kosho_setteibi":"日付","kohosha_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).drop(["社数カウント"], axis=1)
honkosho_settei_company["value"] = honkosho_settei_company.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_settei_company['type'] = 'settei_com'

#本交渉実施だけのテーブル
honkosho_jisshi = honkosho_data[['id','kosho_jisshibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉実施週']].copy()
honkosho_jisshi = honkosho_jisshi.rename(columns={"kosho_jisshibi":"日付","kohosha_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).dropna(subset=["日付"])
honkosho_jisshi["value"] = honkosho_jisshi.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_jisshi['type'] = 'jissi'

honkosho_jisshi_kohosha = honkosho_data[['id','kosho_jisshibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉実施週','候補者重複']].copy()
honkosho_jisshi_kohosha = honkosho_jisshi_kohosha.dropna(subset=["kosho_jisshibi"])
honkosho_jisshi_kohosha = honkosho_jisshi_kohosha.loc[honkosho_jisshi_kohosha["候補者重複"] == 1]
honkosho_jisshi_kohosha = honkosho_jisshi_kohosha.rename(columns={"kosho_jisshibi":"日付","kohosha_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).drop(["候補者重複"], axis=1)
honkosho_jisshi_kohosha["value"] = honkosho_jisshi_kohosha.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_jisshi_kohosha['type'] = 'jissi_kohosha'

honkosho_jisshi_company = honkosho_data[['id','kosho_jisshibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉実施週','社数カウント']].copy()
honkosho_jisshi_company = honkosho_jisshi_company.dropna(subset=["kosho_jisshibi"])
honkosho_jisshi_company = honkosho_jisshi_company.loc[honkosho_jisshi_company["社数カウント"] == 1]
honkosho_jisshi_company = honkosho_jisshi_company.rename(columns={"kosho_jisshibi":"日付","kohosha_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).drop(["社数カウント"], axis=1)
honkosho_jisshi_company["value"] = honkosho_jisshi_company.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_jisshi_company['type'] = 'jissi_com'

honkosho_jisshi_shinki = honkosho_data[['id','kosho_jisshibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉実施週']].copy()
honkosho_jisshi_shinki = honkosho_jisshi_shinki.rename(columns={"kosho_jisshibi":"日付","kohosha_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).dropna(subset=["日付"])
honkosho_jisshi_shinki["value"] = honkosho_jisshi_shinki.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_jisshi_shinki['type'] = 'jissi_shinki'

honkosho_jisshi_sai = honkosho_data[['id','kosho_jisshibi','kohosha_tanto','候_ロンザン所属フラグ','候_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉実施週']].copy()
honkosho_jisshi_sai = honkosho_jisshi_sai.rename(columns={"kosho_jisshibi":"日付","kohosha_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'候_略氏名':'sei_plus'}).dropna(subset=["日付"])
honkosho_jisshi_sai["value"] = honkosho_jisshi_sai.apply(lambda x : 1 if (x["sai_flg2"]  ==2) and (x["候_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_jisshi_sai['type'] = 'jissi_sai'


In [186]:
honkosho_settei.head(10)
honkosho_settei.to_excel('honkosho_settei.xlsx',sheet_name='new_sheet_name')

In [187]:
#企業担当側
#本交渉設定だけのテーブル
honkosho_kigyo_settei = honkosho_data[['id','kosho_setteibi','kigyo_tanto','企_ロンザン所属フラグ','企_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉設定週']].copy()
honkosho_kigyo_settei = honkosho_kigyo_settei.rename(columns={"kosho_setteibi":"日付","kigyo_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'})
honkosho_kigyo_settei["value"] = honkosho_kigyo_settei.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["企_ロンザン所属フラグ"] == 1)  else 0 ,axis = 1)
honkosho_kigyo_settei['type'] = 'kigyo_settei'

honkosho_kigyo_settei_kohosha = honkosho_data[['id','kosho_setteibi','kigyo_tanto','企_ロンザン所属フラグ','企_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉設定週','候補者重複']].copy()
honkosho_kigyo_settei_kohosha = honkosho_kigyo_settei_kohosha.loc[honkosho_kigyo_settei_kohosha["候補者重複"] == 1]
honkosho_kigyo_settei_kohosha = honkosho_kigyo_settei_kohosha.rename(columns={"kosho_setteibi":"日付","kigyo_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).drop(["候補者重複"], axis=1)
honkosho_kigyo_settei_kohosha["value"] = honkosho_kigyo_settei_kohosha.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["企_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_kigyo_settei_kohosha['type'] = 'kigyo_settei_kohosha'

honkosho_kigyo_settei_company = honkosho_data[['id','kosho_setteibi','kigyo_tanto','企_ロンザン所属フラグ','企_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉設定週','社数カウント']].copy()
honkosho_kigyo_settei_company = honkosho_kigyo_settei_company.loc[honkosho_kigyo_settei_company["社数カウント"] == 1]
honkosho_kigyo_settei_company = honkosho_kigyo_settei_company.rename(columns={"kosho_setteibi":"日付","kigyo_tanto":"担当者","本交渉設定週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).drop(["社数カウント"], axis=1)
honkosho_kigyo_settei_company["value"] = honkosho_kigyo_settei_company.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["企_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_kigyo_settei_company['type'] = 'kigyo_settei_com'

#本交渉実施だけのテーブル
honkosho_kigyo_jisshi = honkosho_data[['id','kosho_jisshibi','kigyo_tanto','企_ロンザン所属フラグ','企_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉実施週']].copy()
honkosho_kigyo_jisshi = honkosho_kigyo_jisshi.rename(columns={"kosho_jisshibi":"日付","kigyo_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).dropna(subset=["日付"])
honkosho_kigyo_jisshi["value"] = honkosho_kigyo_jisshi.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["企_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_kigyo_jisshi['type'] = 'kigyo_jissi'

honkosho_kigyo_jisshi_kohosha = honkosho_data[['id','kosho_jisshibi','kigyo_tanto','企_ロンザン所属フラグ','企_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉実施週','候補者重複']].copy()
honkosho_kigyo_jisshi_kohosha = honkosho_kigyo_jisshi_kohosha.dropna(subset=["kosho_jisshibi"])
honkosho_kigyo_jisshi_kohosha = honkosho_kigyo_jisshi_kohosha.loc[honkosho_kigyo_jisshi_kohosha["候補者重複"] == 1]
honkosho_kigyo_jisshi_kohosha = honkosho_kigyo_jisshi_kohosha.rename(columns={"kosho_jisshibi":"日付","kigyo_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).drop(["候補者重複"], axis=1)
honkosho_kigyo_jisshi_kohosha["value"] = honkosho_kigyo_jisshi_kohosha.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["企_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_kigyo_jisshi_kohosha['type'] = 'kigyo_jissi_kohosha'

honkosho_kigyo_jisshi_company = honkosho_data[['id','kosho_jisshibi','kigyo_tanto','企_ロンザン所属フラグ','企_略氏名','APソース丸め','sai_flg2','kaisu','組手','本交渉実施週','社数カウント']].copy()
honkosho_kigyo_jisshi_company = honkosho_kigyo_jisshi_company.dropna(subset=["kosho_jisshibi"])
honkosho_kigyo_jisshi_company = honkosho_kigyo_jisshi_company.loc[honkosho_kigyo_jisshi_company["社数カウント"] == 1]
honkosho_kigyo_jisshi_company = honkosho_kigyo_jisshi_company.rename(columns={"kosho_jisshibi":"日付","kigyo_tanto":"担当者","本交渉実施週":"ｶﾚﾝﾀﾞｰ週",'企_略氏名':'sei_plus'}).drop(["社数カウント"], axis=1)
honkosho_kigyo_jisshi_company["value"] = honkosho_kigyo_jisshi_company.apply(lambda x : 1 if (x["sai_flg2"]  in [0, 1, 2]) and (x["企_ロンザン所属フラグ"] == 1) else 0 ,axis = 1)
honkosho_kigyo_jisshi_company['type'] = 'kigyo_jissi_com'


In [188]:
#本交渉設定と実施をそれぞれ同じ形で整える
honkosho = pd.concat([honkosho_settei,honkosho_settei_kohosha],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_settei_company],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_settei_shinki],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_settei_sai],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_jisshi],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_jisshi_kohosha],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_jisshi_company],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_jisshi_shinki],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_jisshi_sai],ignore_index = True)


honkosho = pd.concat([honkosho,honkosho_kigyo_settei],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_settei_kohosha],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_settei_company],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_jisshi],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_jisshi_kohosha],ignore_index = True)
honkosho = pd.concat([honkosho,honkosho_kigyo_jisshi_company],ignore_index = True)

#Q情報　追加
honkosho = pd.merge(honkosho,Q_master,on = ("日付"),how = "left")
honkosho.head(3)

honkosho.to_excel('honkosho.xlsx',sheet_name='new_sheet_name')


# 紹介受領

In [189]:
sql ="""
select
a.id,
a.kohosha_id,
kohosha_sql.name as kohosha_rank,
b.not_count as kohosha_not_count,
apsource_sql.name as kohosha_apsource,
a.shokaimoto_id,
-- case when a.ap_source = "20" then c.id ELSE NULL END AS `人事部id`,
-- case when a.ap_source = "20" then c.company_name ELSE NULL END AS `会社名`,
-- case when a.ap_source = "20" then c.yakushoku ELSE NULL END AS `役職`,
-- case when a.ap_source = "10" then d.komon_id else null end as "顧問id",
-- case when a.ap_source = "10" then d.company_name else null end as "紹介元企業名",
a.juryosha,
a.juryobi
from
`r-group-bigdata.live_rhs.shokaijuryos` as a

left join
(select id,seimei,ap_source,kohosha_rank,not_count,shokaimoto_id from `r-group-bigdata.live_rhs.kohoshas`) as b ON a.kohosha_id = b.id
left join
(select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 19) as apsource_sql ON a.ap_source = apsource_sql.code
left join
(select group_code,code,name from `r-group-bigdata.live_rhs.sys_consts` where group_code = 5200) as kohosha_sql ON b.kohosha_rank = kohosha_sql.code
-- left join
-- (select id,company_name,yakushoku from `r-group-bigdata.live_rhs.jinjibus`) as c on b.shokaimoto_id = c.id
-- left join
-- (select id,komon_id,company_name from `r-group-bigdata.live_rhs.partners`) as d on a.shokaimoto_id = d.id

where a.juryobi >= '2017-04-01'
;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
shokaijuryos_data = client.query(sql).result().to_dataframe()

#候補者ランクが空欄の箇所を埋める
shokaijuryos_data["kohosha_rank"] =shokaijuryos_data["kohosha_rank"].fillna(0)

#候補者APソースが空欄の行を削除
shokaijuryos_data = shokaijuryos_data.dropna(subset=["kohosha_apsource"])

shokaijuryos_data["juryobi"] = pd.to_datetime(shokaijuryos_data["juryobi"]) #日付データを変換

### 紹介受領データを整える

In [190]:
#候補者APソース　追加
shokaijuryos_data = shokaijuryos_data.rename(columns={"kohosha_apsource":"ヨミ表選択"}) #カラム名変換
shokaijuryos_data = pd.merge(shokaijuryos_data,master4,on = ("ヨミ表選択"),how = "left") #候補者APソースを紐付け
shokaijuryos_data = shokaijuryos_data.rename(columns={"丸め":"APソース丸め"})


In [191]:
#紹介受領週（カレンダー用）　追加
shokaijuryos_data = shokaijuryos_data.rename(columns={"juryobi":"日付"}) #カラム名変換
shokaijuryos_data = pd.merge(shokaijuryos_data,master1,on = ("日付"),how = "left") #初期交渉設定日関連の日付データを紐付け
shokaijuryos_data = shokaijuryos_data.rename(columns={"週":"紹介受領週","計上Q":"Q","日付":"juryobi","juryosha": "user_id"})

shokaijuryos_data = pd.merge(shokaijuryos_data,syain_data,on = "user_id",how="left") #名前が変換される

In [192]:
shokaijuryos_data = shokaijuryos_data[['id','juryobi','user_id','sei_plus','APソース丸め','紹介受領週']].copy()
shokaijuryos_data = shokaijuryos_data.rename(columns={"juryobi":"日付","user_id":"担当者","紹介受領週":"ｶﾚﾝﾀﾞｰ週"})



In [193]:
shokaijuryos_data["value"] = 1
shokaijuryos_data['type'] = 'shokaijuryos'
shokaijuryos_data['組手'] = ''
shokaijuryos_data['sai_flg'] = ''
shokaijuryos_data['kaisu'] = 1
shokaijuryos_data['saikosho_apsource'] = shokaijuryos_data['APソース丸め']

#Q情報　追加
shokaijuryos = pd.merge(shokaijuryos_data,Q_master,on = ("日付"),how = "left")
shokaijuryos.head()

shokaijuryos.to_excel('shokaijuryos.xlsx',sheet_name='new_sheet_name')


In [194]:
shokaijuryos_data

,id,日付,担当者,sei_plus,APソース丸め,ｶﾚﾝﾀﾞｰ週,value,type,組手,sai_flg,kaisu,saikosho_apsource
0,14700,2026-07-30,m-takamine,高峯美,人事部紹介,7月5W(27日-31日),1,shokaijuryos,,,1,人事部紹介
1,6190,2020-03-26,yoshimi,吉見和,人事部紹介,,1,shokaijuryos,,,1,人事部紹介
2,12395,2024-05-20,m-masui,増井真,人事部紹介,,1,shokaijuryos,,,1,人事部紹介
3,14414,2026-03-12,a-maehara,前原杏,パートナー紹介,,1,shokaijuryos,,,1,パートナー紹介
4,13536,2025-06-09,a-maehara,前原杏,パートナー紹介,,1,shokaijuryos,,,1,パートナー紹介
...,...,...,...,...,...,...,...,...,...,...,...,...
13182,12122,2024-02-29,yuu-takahashi,高橋優２,パートナー紹介,,1,shokaijuryos,,,1,パートナー紹介
13183,12776,2024-08-29,yuu-takahashi,高橋優２,パートナー紹介,,1,shokaijuryos,,,1,パートナー紹介
13184,13996,2025-12-05,yuu-takahashi,高橋優２,パートナー紹介,,1,shokaijuryos,,,1,パートナー紹介
13185,12782,2024-08-30,yuu-takahashi,高橋優２,パートナー紹介,,1,shokaijuryos,,,1,パートナー紹介


# 企業AP・企業営業

In [195]:
#営業AP数のデータを出す
sql = """
with base as (
SELECT
ap.id,
ap.kigyo_id,
ap.company_name,
ap.appoint_source,
ap.appoint_get_syain,
ap.appoint_visit_syain,
ap.appoint_get_date,
ap.appoint_visit_plan_date,
ful.fulfills_date,
ful.visit_times as kaisu,
visit_times_auto,
row_number() over(partition by ap.kigyo_id order by appoint_get_date asc) rn
FROM `r-group-bigdata.live_rhs.sales_appoints` ap
left join `r-group-bigdata.live_rhs.sales_appoint_fulfills` ful on ap.id = ful.id
LEFT JOIN (SELECT
            yyyymmdd as date,
            concat(ki,"-",q,"Q") as Q
            FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`) cal on ap.appoint_visit_plan_date = cal.date
where appoint_get_date >= '2017-10-01'
# and appoint_get_syain = "ka-ooki" 
)

select *
from base
where 
-- rn = 1

visit_times_auto = 0 or 
visit_times_auto = 1




;
"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
sales_ap_data = client.query(sql).result().to_dataframe()

sales_ap_data["appoint_get_date"] = pd.to_datetime(sales_ap_data["appoint_get_date"]) #日付データを変換
sales_ap_data["appoint_visit_plan_date"] = pd.to_datetime(sales_ap_data["appoint_visit_plan_date"]) #日付データを変換


In [196]:
#企業初期商談設定週（カレンダー用）　追加
sales_ap_data = sales_ap_data.rename(columns={"appoint_get_date":"日付"}) #カラム名変換
sales_ap_data = pd.merge(sales_ap_data,master1,on = ("日付"),how = "left") #対面商談設定日関連の日付データを紐付け
sales_ap_data = sales_ap_data.rename(columns={"週":"対面商談設定週","日付":"appoint_get_date","appoint_get_syain":"user_id","計上Q":"Q"})
sales_ap_data = pd.merge(sales_ap_data,syain_data,on = "user_id",how="left") #名前が変換される

In [197]:
sales_ap_data = sales_ap_data[['id','kigyo_id','appoint_get_date','user_id',"sei_plus",'対面商談設定週','kaisu']].copy()
sales_ap_data = sales_ap_data.rename(columns={"appoint_get_date":"日付","user_id":"担当者","対面商談設定週":"ｶﾚﾝﾀﾞｰ週"})

In [198]:
sales_ap_data["kaisu"] =sales_ap_data["kaisu"].fillna(0)
# sales_ap_data["value"] = sales_ap_data.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
sales_ap_data["value"] = 1
sales_ap_data['type'] = 'eigyou_ap'
sales_ap_data['組手'] = ''
sales_ap_data['sai_flg'] = ''
sales_ap_data['APソース丸め'] = ''

In [199]:
#Q情報　追加
sales_ap = pd.merge(sales_ap_data,Q_master,on = ("日付"),how = "left")


In [200]:
# sales_ap.to_excel('sales_ap.xlsx',sheet_name='new_sheet_name')

In [201]:
#営業実施数のデータを抽出

sql = """
with base as (
SELECT
ap.id,
ap.kigyo_id,
ap.company_name,
ap.appoint_source,
ap.appoint_get_syain,
ap.appoint_visit_syain,
ap.appoint_get_date,
ap.appoint_visit_plan_date as fulfills_date,
ful.visit_times as kaisu,
visit_times_auto,
row_number() over(partition by ap.kigyo_id order by appoint_get_date asc) rn
FROM `r-group-bigdata.live_rhs.sales_appoints` ap
left join `r-group-bigdata.live_rhs.sales_appoint_fulfills` ful on ap.id = ful.id
LEFT JOIN (SELECT
            yyyymmdd as date,
            concat(ki,"-",q,"Q") as Q
            FROM `r-group-bigdata.live_sugarcrm52.calendar_suka_master`) cal on ap.appoint_visit_plan_date = cal.date
where fulfills_date >= '2017-10-01'
-- and appoint_get_syain = "ka-ooki" 
)

select *
from base
where 
-- rn = 1
visit_times_auto = 0 or visit_times_auto = 1

;

"""
client = bigquery.Client(credentials=credentials, project=credentials.project_id)
sales_apful_data = client.query(sql).result().to_dataframe()

sales_apful_data["fulfills_date"] = pd.to_datetime(sales_apful_data["fulfills_date"]) #日付データを変換

In [202]:
#企業営業実施週（カレンダー用）　追加
sales_apful_data = sales_apful_data.rename(columns={"fulfills_date":"日付"}) #カラム名変換
sales_apful_data = pd.merge(sales_apful_data,master1,on = ("日付"),how = "left") #対面商談設定日関連の日付データを紐付け
sales_apful_data = sales_apful_data.rename(columns={"週":"アポ実施週","日付":"fulfills_date","appoint_visit_syain":"user_id"})
sales_apful_data = pd.merge(sales_apful_data,syain_data,on = "user_id",how="left") #名前が変換される

sales_apful_data = sales_apful_data[['id','fulfills_date','user_id',"sei_plus",'アポ実施週','kaisu']].copy()
sales_apful_data = sales_apful_data.rename(columns={"fulfills_date":"日付","user_id":"担当者","アポ実施週":"ｶﾚﾝﾀﾞｰ週"})

sales_apful_data["kaisu"] =sales_apful_data["kaisu"].fillna(0)

# sales_apful_data["value"] = sales_apful_data.apply(lambda x : 1 if x["kaisu"] == 1 else 0 ,axis = 1)
sales_apful_data["value"] = 1
sales_apful_data['type'] = 'eigyou'
sales_apful_data['組手'] = ''
sales_apful_data['sai_flg'] = ''
sales_apful_data['APソース丸め'] = ''
sales_apful_data['saikosho_apsource'] = ''

#Q情報　追加
sales_apful = pd.merge(sales_apful_data,Q_master,on = ("日付"),how = "left")

# sales_apful.to_excel('sales_apful.xlsx',sheet_name='new_sheet_name')

In [203]:
honkosho = honkosho.rename(columns={"sai_flg2":"sai_flg"})

In [204]:
df = pd.concat([shokikosho, honkosho])
df = pd.concat([df, shokaijuryos])
df = pd.concat([df, sales_ap])
df = pd.concat([df, sales_apful])

# df.to_excel('df.xlsx',sheet_name='new_sheet_name')


In [205]:
#　ロンザン事業部全体数値用 (初接触時のアポソース別)
df_all = df[df["期間対象外"] == "対象"]
df1 = df_all.pivot_table(index=["type","APソース丸め"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_all.pivot_table(index=["type","APソース丸め"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_all.pivot_table(index=["type","APソース丸め"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
# df4 = df_all.pivot_table(index=["type","APソース丸め"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_all=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_all["type"] = concat_all["type"]+concat_all["APソース丸め"]

In [206]:

#　ロンザン事業部全体数値用 (shokikosho　の　再交渉分だけカウント)
df_sai_shoki = df[(df["sai_flg"] == 2) & (df["期間対象外"] == "対象") & (df["type"].str.contains('shoki'))]

df1 = df_sai_shoki.pivot_table(index=["type","sai_flg"],columns="月",aggfunc="count",values="value").fillna(0)
df2 = df_sai_shoki.pivot_table(index=["type","sai_flg"],columns="Q",aggfunc="count",values="value").fillna(0)
df3 = df_sai_shoki.pivot_table(index=["type","sai_flg"],columns="Q同営",aggfunc="count",values="value").fillna(0)
# df4 = df_sai_shoki.pivot_table(index=["type","sai_flg"],columns="Q同旬",aggfunc="count",values="value").fillna(0)
concat_sai_shoki=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_sai_shoki["type"] = concat_sai_shoki["type"] + "_sai" + concat_sai_shoki["sai_flg"].astype(str)

#　ロンザン事業部全体数値用 (honkosho　の　再交渉部分だけカウント)
df_sai_honkosho = df[(df["sai_flg"] == 2) & (df["期間対象外"] == "対象")  & (~df["type"].str.contains('shoki'))]

df1 = df_sai_honkosho.pivot_table(index=["type","sai_flg"],columns="月",aggfunc="count",values="value").fillna(0)
df2 = df_sai_honkosho.pivot_table(index=["type","sai_flg"],columns="Q",aggfunc="count",values="value").fillna(0)
df3 = df_sai_honkosho.pivot_table(index=["type","sai_flg"],columns="Q同営",aggfunc="count",values="value").fillna(0)
# df4 = df_sai_honkosho.pivot_table(index=["type","sai_flg"],columns="Q同旬",aggfunc="count",values="value").fillna(0)
concat_sai_honkosho=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_sai_honkosho["type"] = concat_sai_honkosho["type"]+"_sai"+ concat_sai_honkosho["sai_flg"].astype(str)

# ロンザン事業部全体数値用（組手別）
df_kumite = df[df["期間対象外"] == "対象"]
df1 = df_kumite.pivot_table(index=["type","組手"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_kumite.pivot_table(index=["type","組手"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_kumite.pivot_table(index=["type","組手"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
# df4 = df_kumite.pivot_table(index=["type","組手"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_kumite=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_kumite["type"] = concat_kumite["type"]+concat_kumite["組手"]

concat_sai_shoki = concat_sai_shoki.rename(columns={'sai_flg': 'APソース丸め'})

concat_all = pd.concat([concat_all, concat_sai_shoki], axis=0, ignore_index=True)

# 'sai_flg' 列を文字列型に変換
concat_sai_honkosho = concat_sai_honkosho.rename(columns={'sai_flg': 'APソース丸め'})
concat_all = pd.concat([concat_all, concat_sai_honkosho], axis=0, ignore_index=True)

concat_all = pd.concat([concat_all, concat_kumite], axis=0, ignore_index=True)


In [207]:
# --- メインの変換処理 ---
# DataFrameのすべての列をループで確認します。
for col in concat_all.columns:
    # 列のデータ型（dtype）が 'object' かどうかを判定します。
    if concat_all[col].dtype == 'object':
        # 'object' 型の列を数値に変換しようと試みます。
        # errors='coerce' は、変換できない値を NaN (Not a Number) にするオプションです。
        # これにより、文字が混ざっていてもエラーで止まることを防ぎます。
        converted_series = pd.to_numeric(concat_all[col], errors='coerce')
        
        # 変換後のデータに、一つでも数値（NaNでないもの）が含まれているかを確認します。
        # すべてがNaNになってしまう場合は、元々数値データではないと判断し、変換しません。
        if not converted_series.isnull().all():
            print(f"列 '{col}' を 'object' から 'float64' に変換します。")
            concat_all[col] = converted_series
        else:
            print(f"列 '{col}' は数値に変換できないため、'object' のままにします。")

列 'type' は数値に変換できないため、'object' のままにします。
列 'APソース丸め' を 'object' から 'float64' に変換します。
列 '組手' は数値に変換できないため、'object' のままにします。


In [208]:
#　個人別数値用
df_kojin = df[df["期間対象外"] == "対象"]
df1 = df_kojin.pivot_table(index=["type","sei_plus"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_kojin.pivot_table(index=["type","sei_plus"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_kojin.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
# df4 = df_kojin.pivot_table(index=["type","sei_plus"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_kojin["type"] = concat_kojin["type"]+concat_kojin["sei_plus"]

#　個人別数値用（組手別）
df_kojin_kumite = df[df["期間対象外"] == "対象"]
df1 = df_kojin_kumite.pivot_table(index=["type","組手","sei_plus"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_kojin_kumite.pivot_table(index=["type","組手","sei_plus"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_kojin_kumite.pivot_table(index=["type","組手","sei_plus"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
# df4 = df_kojin_kumite.pivot_table(index=["type","組手","sei_plus"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_kojin_kumite=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_kojin_kumite["type"] = concat_kojin_kumite["type"]+concat_kojin_kumite["組手"]+concat_kojin_kumite["sei_plus"]

#　個人別数値用（紹介アポ獲得数（人））
df_kojin_shoki_ap_partner = df[(df["APソース丸め"] == "パートナー紹介") & (df["期間対象外"] == "対象") & (df["type"].str.contains('shoki_ap'))]
df1 = df_kojin_shoki_ap_partner.pivot_table(index=["type","sei_plus"],columns="月",aggfunc="sum",values="value").fillna(0)
df2 = df_kojin_shoki_ap_partner.pivot_table(index=["type","sei_plus"],columns="Q",aggfunc="sum",values="value").fillna(0)
df3 = df_kojin_shoki_ap_partner.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="sum",values="value").fillna(0)
# df4 = df_kojin_shoki_ap_partner.pivot_table(index=["type","sei_plus"],columns="Q同旬",aggfunc="sum",values="value").fillna(0)
concat_kojin_shoki_ap_partner=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_kojin_shoki_ap_partner["type"] = concat_kojin_shoki_ap_partner["type"] + "_partner" +concat_kojin_shoki_ap_partner["sei_plus"]

#　ロンザン事業部全体数値用 (shokikosho　の　再交渉分だけカウント)
df_kojin_shoki = df[(df["sai_flg"] == 2) & (df["期間対象外"] == "対象") & (df["type"].str.contains('shoki'))]
df1 = df_sai_shoki.pivot_table(index=["type","sai_flg","sei_plus"],columns="月",aggfunc="count",values="value").fillna(0)
df2 = df_sai_shoki.pivot_table(index=["type","sai_flg","sei_plus"],columns="Q",aggfunc="count",values="value").fillna(0)
df3 = df_sai_shoki.pivot_table(index=["type","sai_flg","sei_plus"],columns="Q同営",aggfunc="count",values="value").fillna(0)
# df4 = df_sai_shoki.pivot_table(index=["type","sai_flg","sei_plus"],columns="Q同旬",aggfunc="count",values="value").fillna(0)
concat_kojin_shoki=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_kojin_shoki["type"] = concat_kojin_shoki["type"]+"_sai"+concat_kojin_shoki["sei_plus"]

concat_kojin = pd.merge(concat_kojin,concat_kojin_kumite,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_kojin_shoki_ap_partner,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_kojin_shoki,how = "outer")

In [209]:
#  カレンダー用
#アポソース別（再交渉になった案件は、アポソース別の数値から除く　※初回が人事部、その後再交渉案件の場合は、　人事部のカウント0、再交渉側のアポソースで1）
df_calendar = df[(df["期間対象外"] == "対象") & (df["sai_flg"] != 2)]
calendar = df_calendar.pivot_table(index=["type","APソース丸め","sei_plus","sai_flg"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="sum",values="value").fillna(0).reset_index()
calendar["type"] = calendar["type"]+calendar["APソース丸め"]+calendar["sei_plus"]

#再交渉用（再交渉になったアポソースのものをカウント　※初回が人事部、その後再交渉案件の場合は、　人事部のカウント0、再交渉側のアポソースで1）
df_calendar_sai = df[(df["sai_flg"] == 2) & (df["期間対象外"] == "対象") & (df["kaisu"] == 1)]
calendar_sai = df_calendar_sai.pivot_table(index=["type","sei_plus","sai_flg"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="count",values="value").fillna(0).reset_index()
calendar_sai["type"] = calendar_sai["type"]+"_sai"+calendar_sai["sei_plus"]

#再交渉用（再交渉になったアポソースのものをカウント　shokiko部分のみカウント）
df_calendar_sai_shoki = df[(df["sai_flg"] == 2) & (df["期間対象外"] == "対象") & (df["type"].str.contains('shoki'))]
calendar_sai_shoki = df_calendar_sai_shoki.pivot_table(index=["type","sei_plus","sai_flg"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="count",values="value").fillna(0).reset_index()
calendar_sai_shoki["type"] = calendar_sai_shoki["type"]+"_sai"+calendar_sai_shoki["sei_plus"]


#企業設定・実施用
df_calendar_kigyo = df[df["期間対象外"] == "対象"]
calendar_kigyo = df_calendar_kigyo.pivot_table(index=["type","sei_plus"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="sum",values="value").fillna(0).reset_index()
calendar_kigyo["type"] = calendar_kigyo["type"]+"_all_1"+calendar_kigyo["sei_plus"]

#企業設定・実施用(2回目をカウント)
df_calendar_kigyo_jissi = df[(df["kaisu"] == 2) & (df["期間対象外"] == "対象") ]
calendar_kigyo_jissi = df_calendar_kigyo_jissi.pivot_table(index=["type","sei_plus"],columns="ｶﾚﾝﾀﾞｰ週",aggfunc="count",values="value").fillna(0).reset_index()
calendar_kigyo_jissi["type"] = calendar_kigyo_jissi["type"]+"_all_2"+calendar_kigyo_jissi["sei_plus"]

calendar = pd.merge(calendar_sai,calendar,how = "outer")
calendar = pd.merge(calendar,calendar_sai_shoki,how = "outer")
calendar = pd.merge(calendar,calendar_kigyo,how = "outer")
calendar = pd.merge(calendar,calendar_kigyo_jissi,how = "outer")

# 顧客支持pt、受注数 

In [210]:
# 過去ヨミ表のA~E列を取得
SPREADSHEET_ID = "1ITzx2eIAMpiepjGVrDSf-FR1XcAKzYJMqFZuS6-8qb0"
Sheet_NAME = 'data!'
Sheet_row = "A:E"
RANGE_NAME = Sheet_NAME+Sheet_row
yomi_old_betu = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

yomi_old_betu = yomi_old_betu.rename(columns={"両手案件フラグ":"旧両手",
                                              "アポソース丸め":"旧アポ",
                                              "候補者アポソース":"旧候補者アポ",
                                              "候補者id":"旧候id"})

yomi_old_betu["発番"] = pd.to_numeric(yomi_old_betu["発番"], errors='coerce')

# 過去ヨミ表のA~E列を取得
SPREADSHEET_ID = "1ITzx2eIAMpiepjGVrDSf-FR1XcAKzYJMqFZuS6-8qb0"
Sheet_NAME = 'data!'
Sheet_row = "G:AV"
RANGE_NAME = Sheet_NAME+Sheet_row
yomi_old = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

yomi_old = yomi_old.rename(columns={"営業売上\n（営業ポイント）.1":"営業売上\n（営業ポイント）"})

In [211]:
# 今Qのヨミ表を取得
SPREADSHEET_ID = "1h8tyDhieP_gVp2Enj6dEJ3iR5yjyJzYNfLz1FH8U7r8"
Sheet_NAME = 'ヨミ表!'
Sheet_row = "A10:AZ"
RANGE_NAME = Sheet_NAME+Sheet_row
yomi_nowQ = ps.get_ss(SPREADSHEET_ID,RANGE_NAME,service)

In [212]:
yomi_nowQ["計上日"] = pd.to_datetime(yomi_nowQ["計上日"])
yomi_nowQ["計上月"] = yomi_nowQ["計上日"].dt.month
yomi_nowQ["報酬率"] = pd.to_numeric(yomi_nowQ["報酬率"].str.replace('%', '', regex=False), errors='coerce') / 100

#RPA事業部のクロスセルは除く(顧問名が「塩澤昌紘」はRPA事業部のクロスセル案件)
yomi_nowQ = yomi_nowQ[~yomi_nowQ['候補者'].isin(['塩澤昌紘'])]
yomi_nowQ = yomi_nowQ[~yomi_nowQ['売上種別（商品内容）'].isin(['RPAコンサル'])]
yomi_nowQ = yomi_nowQ[~yomi_nowQ['売上種別（商品内容）'].isin(['ビジネスタンク'])]
# yomi_nowQ.to_excel('yomi_nowQ.xlsx')

yomi_nowQ["入力者"] = ""
yomi_nowQ["グループ企画料"] = ""
# yomi_nowQ['貢献引当後pt'] = ""
yomi_nowQ['特殊フラグ'] = ""
yomi_nowQ['備考①'] = ""
yomi_nowQ['備考②'] = ""
yomi_nowQ['備考③'] = ""
yomi_nowQ['担当\n押印'] = ""
yomi_nowQ['d'] = ""
yomi_nowQ['d.1'] = ""
yomi_nowQ['アポソース'] = ""

yomi_nowQ = yomi_nowQ.rename(columns={'案件id （RZ）': '案件id\n（RZ）',
                                      '案件No （SC）':'案件\nNo\n（SC）',
                                      '完保成約割振比':'完保\n成約\n割振比',
                                      'サービス引当係数':'サービス\n引当\n係数',
                                      'キャンセル引当係数':'キャンセル\n引当\n係数',
                                      '査定用売上':'査定用\n売上',
                                      'カテゴリ':'担当',
                                      'シニアスカウト （ヨミ表と一致）':'シニアスカウト\n（ヨミ表と一致）',
                                      })

yomi_nowQ = yomi_nowQ[['案件id（RZ）','計上月', '期', '月','案件No（SC）','入力者','計上日','受注日',
                       'クライアント正式名称','候補者','売上種別（商品内容）','差分\n（該当場合のみ）',
                       '紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）','基準年収', '報酬率', '完保\n成約\n割振比',"グループ企画料",
                       'サービス\n引当\n係数', 'キャンセル\n引当\n係数','営業売上合計', '所属課', '氏名','割合', '受注額',
                       '査定用\n売上','顧客支持ポイント', '貢献引当後pt','担当', '担当\n押印', '備考①', '備考②', '備考③',
                       'シニアスカウト\n（ヨミ表と一致）', '内定数フラグ','企業アポソース','特殊フラグ','提示/前年度','基準年収.1',
                       '報酬率.1', 'd', 'd.1', 'アポソース']]


In [213]:
display(yomi_old.dtypes)
display(yomi_nowQ.dtypes)

案件id（RZ）                       object
計上月                            object
期                              object
計上月                            object
案件No（SC）                       object
入力者                            object
計上日                            object
受注日                            object
クライアント正式名称                     object
候補者                            object
売上種別（商品内容）                     object
差分\n（該当場合のみ）                   object
紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）    object
基準年収                           object
報酬率                            object
完保\n成約\n割振比                    object
グループ企画料                        object
サービス\n引当\n係数                   object
キャンセル\n引当\n係数                  object
営業売上合計                         object
所属課                            object
氏名                             object
割合                             object
受注額                            object
査定用\n売上                        object
顧客支持ポイント                       object
貢献引当後pt     

案件id（RZ）                               object
計上月                                   float64
期                                      object
月                                      object
案件No（SC）                               object
入力者                                    object
計上日                            datetime64[ns]
受注日                                    object
クライアント正式名称                             object
候補者                                    object
売上種別（商品内容）                             object
差分\n（該当場合のみ）                           object
紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）            object
基準年収                                   object
報酬率                                   float64
完保\n成約\n割振比                            object
グループ企画料                                object
サービス\n引当\n係数                           object
キャンセル\n引当\n係数                          object
営業売上合計                                 object
所属課                                    object
氏名                                

In [214]:
# 結合前に、列名の重複を解消する（最初の列だけを残す）
yomi_old = yomi_old.loc[:, ~yomi_old.columns.duplicated()]
yomi_nowQ = yomi_nowQ.loc[:, ~yomi_nowQ.columns.duplicated()]

# 過去計上済のヨミ表情報と、今Qのヨミ表とをくっつける
yomi = pd.concat([yomi_old, yomi_nowQ], axis=0, ignore_index=True)

yomi.loc[yomi['氏名'].isin(['京谷悠子', '京谷悠']), '氏名'] = 'yuko-kyotani'
yomi.loc[yomi['氏名'] == '竹下綾', '氏名'] = 'aya-takeshita'

yomi.columns

Index(['案件id（RZ）', '計上月', '期', '案件No（SC）', '入力者', '計上日', '受注日', 'クライアント正式名称',
       '候補者', '売上種別（商品内容）', '差分\n（該当場合のみ）', '紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）',
       '基準年収', '報酬率', '完保\n成約\n割振比', 'グループ企画料', 'サービス\n引当\n係数',
       'キャンセル\n引当\n係数', '営業売上合計', '所属課', '氏名', '割合', '受注額', '査定用\n売上',
       '顧客支持ポイント', '貢献引当後pt', '担当', '担当\n押印', '備考①', '備考②', '備考③',
       'シニアスカウト\n（ヨミ表と一致）', '内定数フラグ', '企業アポソース', '特殊フラグ', '提示/前年度', '基準年収.1',
       '報酬率.1', 'd', 'd.1', 'アポソース', '月'],
      dtype='object')

In [215]:
# 「計上日」カラムを datetime64[ns] 型に変換
yomi['計上日'] = pd.to_datetime(yomi['計上日'], errors='coerce')

# 「受注日」カラムを datetime64[ns] 型に変換
yomi['受注日'] = pd.to_datetime(yomi['受注日'], errors='coerce')

In [216]:
#ポイントデータを結合する
yomi_data =yomi.copy()
yomi_data.index = yomi_data.index + 1
yomi_data = yomi_data.reset_index()
yomi_data = yomi_data.set_index("index")

#上からナンバーを割り振る
yomi_data = yomi_data.reset_index()
yomi_data = yomi_data.rename(columns={"index":"発番"})
#yomi_data.to_excel('総合ポイント.xlsx')

yomi_data = yomi_data.dropna(subset=['顧客支持ポイント'])

# 1. '%'が含まれているデータかどうかを判定する目印（マスク）を作ります
mask = yomi_data["報酬率"].astype(str).str.contains('%', na=False)

# 2. '%'が含まれているデータだけ、'%'を削除して100で割ります（例: 65% -> 0.65）
yomi_data.loc[mask, "報酬率"] = yomi_data.loc[mask, "報酬率"].astype(str).str.replace('%', '', regex=False).astype(float) / 100

# 3. カラム全体を確実に数値データ（float型）に変換します
yomi_data["報酬率"] = pd.to_numeric(yomi_data["報酬率"], errors='coerce')

# yomi_data = yomi_data.rename(columns={"氏名":"sei_plus"})
# yomi_data.to_excel('yomi_data.xlsx',sheet_name='new_sheet_name')
# yomi_data.sample(5)

案件情報（本交渉情報）からアポソースや組手情報などを紐付ける

In [217]:
#設定元データと最新ポイントファイルを結合 (アポソース等を紐付けるため)
honkosho_settei_data = honkosho_data[['id','kosho_setteibi','kohosha_id','APソース丸め','組手','sai_flg','anken_id','hanjokin']].copy()
honkosho_settei_data = honkosho_settei_data.rename(columns={"anken_id":"案件id（RZ）"})
honkosho_settei_data = honkosho_settei_data.drop_duplicates(subset=["案件id（RZ）"],keep='first')
    
yomi_data["案件id（RZ）"]=pd.to_numeric(yomi_data["案件id（RZ）"],errors='coerce')
yomi_data["案件id（RZ）"]=yomi_data["案件id（RZ）"].fillna(0.0).astype(int)
yomi_data["案件id（RZ）"]=yomi_data["案件id（RZ）"].astype(int)

yomi_data = pd.merge(yomi_data,honkosho_settei_data,on = "案件id（RZ）",how="left")

#ヨミ表データと過去のヨミ表データを紐付ける（候補者アポソース取得のため）
yomi_data = pd.merge(yomi_data,yomi_old_betu,on = "発番",how="left")

In [218]:
#過去ポイントファイルの計上日と担当者をマスタデータと紐付けるため、名前を変換
yomi_data = yomi_data.rename(columns={"計上日":"日付","月":"計上日月"})

#カレンダー用の週データを紐付ける
yomi_data["日付"] = pd.to_datetime(yomi_data["日付"])
yomi_data = pd.merge(yomi_data,master1,on = ("日付"),how = "left")
yomi_data = yomi_data.drop(["月"], axis=1).rename(columns={"週":"計上週"})

#Qマスタのデータと紐付ける（同営業日・同旬月日など）
Q_master["日付"] = pd.to_datetime(Q_master["日付"])
yomi_data = pd.merge(yomi_data,Q_master,on = ("日付"),how = "left")

In [219]:
#当時のQの職種を抽出するためにQと担当者名を紐付ける
mas = master2[['Q','sei_plus','職種','ロンザン所属フラグ']].copy()
mas["Q所属フラグ"] = mas["Q"] + mas["sei_plus"]
mas= mas.drop_duplicates(subset=["Q所属フラグ"],keep='first').drop(["Q"], axis=1)

In [220]:
yomi_data["計上Q"] = yomi_data["計上Q"].fillna('').astype(str)
yomi_data["氏名"] = yomi_data["氏名"].fillna('').astype(str)
yomi_data["Q所属フラグ"] = yomi_data["計上Q"] + yomi_data["氏名"]
yomi_data["Q所属フラグ"]=yomi_data["Q所属フラグ"].fillna(0.0)


In [221]:



yomi_data = pd.merge(yomi_data,mas,on = ("Q所属フラグ"),how = "left")



In [222]:
mas.columns

Index(['sei_plus', '職種', 'ロンザン所属フラグ', 'Q所属フラグ'], dtype='object')

In [223]:
yomi_data.columns

Index(['発番', '案件id（RZ）', '計上月', '期', '案件No（SC）', '入力者', '日付', '受注日',
       'クライアント正式名称', '候補者', '売上種別（商品内容）', '差分\n（該当場合のみ）',
       '紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）', '基準年収', '報酬率', '完保\n成約\n割振比', 'グループ企画料',
       'サービス\n引当\n係数', 'キャンセル\n引当\n係数', '営業売上合計', '所属課', '氏名', '割合', '受注額',
       '査定用\n売上', '顧客支持ポイント', '貢献引当後pt', '担当', '担当\n押印', '備考①', '備考②', '備考③',
       'シニアスカウト\n（ヨミ表と一致）', '内定数フラグ', '企業アポソース', '特殊フラグ', '提示/前年度', '基準年収.1',
       '報酬率.1', 'd', 'd.1', 'アポソース', '計上日月', 'id', 'kosho_setteibi',
       'kohosha_id', 'APソース丸め', '組手', 'sai_flg', 'hanjokin', '旧両手', '旧アポ',
       '旧候補者アポ', '旧候id', '計上週', '計上Q', '月', '営業日', 'Q', '同旬月日比較用', '同営業日比較',
       '同旬月日比較', '初中最終月', 'Q月別', 'Q月週別', '月次経過日時', '同旬月日時', '期間対象外', 'Q同営',
       'Q同旬', 'Q所属フラグ', 'sei_plus', '職種', 'ロンザン所属フラグ'],
      dtype='object')

In [224]:
yomi_data

,発番,案件id（RZ）,計上月,期,案件No（SC）,入力者,日付,受注日,クライアント正式名称,候補者,売上種別（商品内容）,差分\n（該当場合のみ）,紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）,基準年収,報酬率,完保\n成約\n割振比,グループ企画料,サービス\n引当\n係数,キャンセル\n引当\n係数,営業売上合計,所属課,氏名,割合,受注額,査定用\n売上,顧客支持ポイント,貢献引当後pt,担当,担当\n押印,備考①,備考②,備考③,シニアスカウト\n（ヨミ表と一致）,内定数フラグ,企業アポソース,特殊フラグ,提示/前年度,基準年収.1,報酬率.1,d,d.1,アポソース,計上日月,id,kosho_setteibi,kohosha_id,APソース丸め,組手,sai_flg,hanjokin,旧両手,旧アポ,旧候補者アポ,旧候id,計上週,計上Q,月,営業日,Q,同旬月日比較用,同営業日比較,同旬月日比較,初中最終月,Q月別,Q月週別,月次経過日時,同旬月日時,期間対象外,Q同営,Q同旬,Q所属フラグ,sei_plus,職種,ロンザン所属フラグ
0,1,0,4月,19,8606,石田,2016-04-15,2016-03-31,株式会社フジダン,白川 正明 氏,スカウト報酬（通常）,,,630,0.55,1,,,,,ロンザン,宮崎佳,0.4,138.6,97.02,82.467,82.467,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,<NA>,NaT,<NA>,NaN,NaN,<NA>,NaN,片手,■若手スカウト,紹介②,None,,19-3Q,2016-04-01,12,19-3Q,1-15,同営業日,同旬月日,初月,19-4月,19-4月-3,12,15,対象,19-3Q同営業日,19-3Q同旬月日,19-3Q宮崎佳,宮崎佳,ミドル企業,1
1,2,0,4月,19,,石田,2016-04-15,2016-04-15,株式会社林間,岩本 昌和 氏,スカウト報酬（通常）,,,740,0.58,1,,,,,ロンザン,宮崎佳,0.4,171.68,120.176,102.1496,102.1496,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,<NA>,NaT,<NA>,NaN,NaN,<NA>,NaN,片手,■若手スカウト,Gアポ,None,,19-3Q,2016-04-01,12,19-3Q,1-15,同営業日,同旬月日,初月,19-4月,19-4月-3,12,15,対象,19-3Q同営業日,19-3Q同旬月日,19-3Q宮崎佳,宮崎佳,ミドル企業,1
2,3,0,4月,19,,石田,2016-04-21,2016-04-21,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,,,1110,0.58,1,643.8,,,,シニア引当,サービス引当,0.1,64.38,64.38,64.38,64.38,,,本採用内定確認書は平成28年1月20日付で締結した人材コンサルティングサービスに関する契約書...,1,B,●,None,None,None,None,None,None,None,None,None,NaN,<NA>,NaT,<NA>,NaN,NaN,<NA>,NaN,片手,顧問,顧問,内定71,,19-3Q,2016-04-01,16,19-3Q,1-21,同営業日,同旬月日,初月,19-4月,19-4月-4,16,21,対象,19-3Q同営業日,19-3Q同旬月日,19-3Qサービス引当,NaN,NaN,NaN
3,4,0,4月,19,,石田,2016-04-21,2016-04-21,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,,,1110,0.58,1,643.8,,,,シニア引当,CXL引当,0.2,128.76,128.76,128.76,128.76,,,,,,●,None,None,None,None,None,None,None,None,None,NaN,<NA>,NaT,<NA>,NaN,NaN,<NA>,NaN,片手,顧問,顧問,内定71,,19-3Q,2016-04-01,16,19-3Q,1-21,同営業日,同旬月日,初月,19-4月,19-4月-4,16,21,対象,19-3Q同営業日,19-3Q同旬月日,19-3QCXL引当,NaN,NaN,NaN
4,5,0,4月,19,,石田,2016-04-21,2016-04-21,株式会社星野産商,鼠入 宏明 氏,シニアスカウト報酬,,,1110,0.58,1,643.8,,,,ロンザン,長崎文,0.2975,191.5305,191.5305,191.5305,191.5305,候補者担当,,,,,●,1,None,None,None,None,None,None,None,None,NaN,<NA>,NaT,<NA>,NaN,NaN,<NA>,NaN,片手,顧問,顧問,内定71,,19-3Q,2016-04-01,16,19-3Q,1-21,同営業日,同旬月日,初月,19-4月,19-4月-4,16,21,対象,19-3Q同営業日,19-3Q同旬月日,19-3Q長崎文,長崎文,ミドル企業,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91658,92425,0,NaN,,None,,NaT,NaT,,,,None,None,,NaN,,,None,None,,,,,None,None,,0.0000,,,,,,None,None,None,,None,None,None,,,,,<NA>,NaT,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN
91659,92426,0,NaN,,None,,NaT,NaT,,,,None,None,,NaN,,,None,None,,,,,None,None,,0.0000,,,,,,None,None,None,,None,None,None,,,,,<NA>,NaT,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN
91660,92427,0,NaN,,None,,NaT,NaT,,,,None,None,,NaN,,,None,None,,,,,None,None,,0.0000,,,,,,None,None,None,,None,None,None,,,,,<NA>,NaT,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN
91661,92428,0,NaN,,None,,NaT,NaT,,,,None,None,,NaN,,,None,None,,,,,None,None,,0.0000,,,,,,None,None,None,,None,None,None,,,,,<NA>,NaT,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN


In [225]:


#企業担当フラグ（この成約の企業担当にフラグを付ける）
yomi_data["企業担当フラグ"] = yomi_data.apply(lambda x : 1 if x["担当"] in 
                                       ("企業担当②","企業担当（本交渉同席）","企業担当（クロージング担当）","面談担当（企業担当）","企業担当（サブ）",
                                        "企業担当戻りP（内定者3名目）","本交渉実施・企業側交渉","企業担当","本交渉担当") else 0 ,axis = 1)
# yomi_data["企業担当フラグ"] = yomi_data["Q所属フラグ_y"].notna().astype(int)

#候補者IDが紐づかない受注については、過去使った候補者IDを紐付ける
yomi_data["kohosha_id"] = yomi_data["kohosha_id"].fillna(0.0)
yomi_data["候補者id"] = yomi_data.apply(lambda x : x["旧候id"] if x["kohosha_id"] == 0 else x["kohosha_id"] ,axis = 1)

#本交渉IDが紐づかない受注については、過去使ったアポソースを紐付ける）
yomi_data["APソース丸め"] = yomi_data["APソース丸め"].fillna(0.0)
yomi_data["候補者APソース2"] = yomi_data.apply(lambda x : x["旧アポ"] if x["APソース丸め"] == 0 else x["APソース丸め"] ,axis = 1)

#ヨミ表のアポソース丸めデータと紐付ける
yomi_data = yomi_data.rename(columns={"候補者APソース2":"ヨミ表選択"})
yomi_data["ヨミ表選択"]=yomi_data["ヨミ表選択"].fillna(0.0)
yomi_data = pd.merge(yomi_data,master4,on = ("ヨミ表選択"),how = "left")
yomi_data["丸め"] = yomi_data["丸め"].fillna(0.0)

yomi_data["候補者アポソース丸め"] = yomi_data.apply(
    lambda x: "その他" if x["売上種別（商品内容）"] == "コンサルティング報酬" 
    else ("■クロスセル" if x["丸め"] == 0 else x["丸め"]), 
    axis=1
)
yomi_data = yomi_data.drop(["丸め"], axis=1)




In [226]:

yomi_data= yomi_data.rename(columns={"所属フラグ":"担当所属フラグ","月":"計上月末","日付マスタ":"計上日"})
yomi_data = yomi_data.drop(['sei_plus'],axis=1)

In [227]:
# 過去計上Qの掛け率などをこちらで設定
yomi_data["計上Q"] = yomi_data["計上Q"].fillna(0.0)
yomi_data = pd.merge(yomi_data, master5, on=("計上Q"), how="left")

# 顧客支持ポイントの処理
yomi_data["顧客支持ポイント"] = pd.to_numeric(yomi_data["顧客支持ポイント"], errors='coerce').fillna(0.0)
yomi_data["貢献引当後pt"] = pd.to_numeric(yomi_data["貢献引当後pt"], errors='coerce').fillna(0.0)


# 掛け率を数値型に変換
yomi_data["掛け率"] = pd.to_numeric(yomi_data["掛け率"], errors='coerce').fillna(0.0)

# ポイント丸めの計算
yomi_data["顧客支持ポイント"]=yomi_data["顧客支持ポイント"].fillna(0.0)

yomi_data["ポイント丸め"] = yomi_data.apply(lambda x: 1 if x["掛け率"] == 0 else x["掛け率"] * x["顧客支持ポイント"], axis=1)


yomi_data["ポイント丸め(組織貢献引当後)"] = yomi_data.apply(lambda x: 1 if x["掛け率"] == 0 else x["掛け率"] * x["貢献引当後pt"], axis=1)
# yomi_data["ポイント丸め"] = yomi_data.apply(lambda x : 1 if x["掛け率"] == "0" else x["掛け率"] * x["顧客支持ポイント"] ,axis = 1)


# 列名の変更
yomi_data = yomi_data.rename(columns={"案件id\n（RZ）": "案件id（RZ）", "シニアスカウト\n（ヨミ表と一致）": "シニアスカウト"})

yomi_data.sample(10)





,発番,案件id（RZ）,計上月,期,案件No（SC）,入力者,日付,受注日,クライアント正式名称,候補者,売上種別（商品内容）,差分\n（該当場合のみ）,紹介引当/PM引当\n特殊ポイント\n（該当場合のみ）,基準年収,報酬率,完保\n成約\n割振比,グループ企画料,サービス\n引当\n係数,キャンセル\n引当\n係数,営業売上合計,所属課,氏名,割合,受注額,査定用\n売上,顧客支持ポイント,貢献引当後pt,担当,担当\n押印,備考①,備考②,備考③,シニアスカウト,内定数フラグ,企業アポソース,特殊フラグ,提示/前年度,基準年収.1,報酬率.1,d,d.1,アポソース,計上日月,id,kosho_setteibi,kohosha_id,APソース丸め,組手,sai_flg,hanjokin,旧両手,旧アポ,旧候補者アポ,旧候id,計上週,計上Q,計上月末,営業日,Q,同旬月日比較用,同営業日比較,同旬月日比較,初中最終月,Q月別,Q月週別,月次経過日時,同旬月日時,期間対象外,Q同営,Q同旬,Q所属フラグ,職種,ロンザン所属フラグ,企業担当フラグ,候補者id,ヨミ表選択,候補者アポソース丸め,修正後ポイント,Q計上時ポイント,掛け率,ポイント丸め,ポイント丸め(組織貢献引当後)
72657,73424,0,NaN,,None,,NaT,NaT,,,,None,None,,NaN,,,None,None,,,,,None,None,0.000000,0.000000,,,,,,None,None,None,,None,None,None,,,,,<NA>,NaT,0,0.0,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,0,NaN,0.0,■クロスセル,NaN,NaN,0.00,1.000000,1.000000
87128,87895,0,NaN,,None,,NaT,NaT,,,,None,None,,NaN,,,None,None,,,,,None,None,0.000000,0.000000,,,,,,None,None,None,,None,None,None,,,,,<NA>,NaT,0,0.0,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,0,NaN,0.0,■クロスセル,NaN,NaN,0.00,1.000000,1.000000
76876,77643,0,NaN,,None,,NaT,NaT,,,,None,None,,NaN,,,None,None,,,,,None,None,0.000000,0.000000,,,,,,None,None,None,,None,None,None,,,,,<NA>,NaT,0,0.0,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,0,NaN,0.0,■クロスセル,NaN,NaN,0.00,1.000000,1.000000
90363,91130,0,NaN,,None,,NaT,NaT,,,,None,None,,NaN,,,None,None,,,,,None,None,0.000000,0.000000,,,,,,None,None,None,,None,None,None,,,,,<NA>,NaT,0,0.0,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,0,NaN,0.0,■クロスセル,NaN,NaN,0.00,1.000000,1.000000
28037,28629,13995,8月,25,15122,加藤,2022-08-22,2022-08-09,株式会社シャトレーゼ,松浦 秀和 氏,シニアスカウト報酬,,,1059,0.65,,,,,688.35,シニア引当,サービス引当,1,688.35,,61.951500,61.951500,サービス引当,,,0,None,None,None,None,None,None,None,None,None,None,None,NaN,14766,2022-06-16,27242,人事部紹介,両手,0,-,両手,人事部紹介,人事部紹介,27242,,25-4Q,2022-08-01,33,25-4Q,2-22,,同旬月日,中月,25-8月,25-8月-1,,,対象,None,None,25-4Qサービス引当,NaN,NaN,0,27242,人事部紹介,人事部紹介,,,1.00,61.951500,61.951500
8347,8711,5207,7月,22,8346,赤松,2019-07-25,2019-07-03,内藤建設株式会社,田中 聡 氏,シニアスカウト報酬,,,1104,0.58,,,,,640.32,シニア引当,CXL引当,1,640.32,,128.064000,128.064000,CXL引当②,,,,0,●,,,,,,,,,SMAP,NaN,5576,2019-06-07,12634,SMAP,両手,1,-,両手,SMAP,SMAP,12634,,22-4Q,2019-07-01,17,22-4Q,1-25,同営業日,同旬月日,初月,22-7月,22-7月-4,,,対象,22-4Q同営業日,22-4Q同旬月日,22-4QCXL引当,NaN,NaN,0,12634,SMAP,SMAP,,,0.86,110.135040,110.135040
27089,27677,0,,,,,NaT,NaT,,,,,,,NaN,,,,,,,,,,,0.000000,0.000000,,,,,,,,,,,,,,,0,NaN,<NA>,NaT,0,0.0,NaN,<NA>,NaN,片手,■クロスセル,0,0,NaN,,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,0,0,■クロスセル,■クロスセル,NaN,NaN,0.00,1.000000,1.000000
31159,31751,15034,2月,26,14984,加藤,2023-02-28,2023-12-26,三洋通商株式会社,城川 和也 氏,シニアスカウト報酬,,,917.556,0.62,,,,,568.88472,SC引当,RP引当,-1,-12.97530214,,-11.210661,-11.210661,,,,,,●,,,,,,,,,転機,NaN,15850,2022-11-22,37778,転機,片手,0,-,片手,転機,転機,37778,,26-2Q,2023-02-01,38,26-2Q,2-28,,同旬月日,中月,26-2月,26-2月-1,,,対象,None,None,26-2QRP引当,NaN,NaN,0,37778,転機,転機,,,1.00,-11.210661,-11.210661
3477,3478,2227,3月,21,,赤松,2018-03-28,2018-03-28,株式会社エンルート,松尾 英明 氏,シニアスカウト報酬,,,1008,0.58,1,,0.7,0.15,584.64,ロンザン,仁木正,1,584.64,409.248,61.387200,61.387200,アポ獲得者,,,,ロンザン,●,,,,,,,,,転機,NaN,2274,2018-01-15,4239,転機,両手,1,-,両手,転機,転機,4239,,21-2Q,2018-03-01,56,21-2Q,3-28,,,最終月,21-3月,21-3月-5,19,28,対象,None,None,21-2Q仁木正,,1,0,4239,転機,転機,"10,638",,0.86,52.792992,52.792992
17891,18477,9355,12月,24,12619,加藤,2020-12-28,2020-12-26,株式会社ラグノオささき,中川 広志 氏,シニアスカウト報酬,,,720,0.65,,,,,468,SC引当,サービス引当,1,468,,46.800000,46.800000,,,,,,,,,,,,,,,パートナー紹介,NaN,9968,2020-11-05,23114,パートナー紹介,片手,1,-,片手,パートナー紹介,,23114,,24-1Q,2020-12-01,59,24-1Q,3-28,,,最終月,24-12月,24-12月-1,,,対象,None,None,24-1Qサービス引当,NaN,NaN,0,23114,パートナー紹介,パートナー紹介,,,1.00,46.800000,46.800000


In [228]:
#両手企業、片手企業のフラグを作成
#候補者担当がRZ所属（退職者含む）、企業担当RZ所属フロントミドル→両手
#候補者担当がRZ所属（退職者含む）以外、企業担当RZ所属フロントミドル→片手

rt1 = yomi_data[(yomi_data["企業担当フラグ"] == 1) & (yomi_data["所属課"] == 'ロンザン') ]
rt1 = rt1.loc[:,["案件id（RZ）","企業担当フラグ","所属課"]]
rt1["案件id（RZ）"] = rt1.apply(lambda x : "-" if x["案件id（RZ）"] == 0 else x["案件id（RZ）"] ,axis = 1)
rt1["企業担当ロンザンか"] = 1.0
rt1 = rt1.loc[:,["案件id（RZ）",'企業担当ロンザンか']]
rt1["案件id（RZ）"] = rt1["案件id（RZ）"].astype(str)
rt1 = rt1.drop_duplicates(subset=["案件id（RZ）"],keep='first')

rt2 = yomi_data[(yomi_data["内定数フラグ"] == 1) & (yomi_data["所属課"] == 'ロンザン') ]
rt2 = rt2.loc[:,["案件id（RZ）","内定数フラグ","所属課"]]
rt2["案件id（RZ）"] = rt2.apply(lambda x : "-" if x["案件id（RZ）"] == 0 else x["案件id（RZ）"] ,axis = 1)
rt2["候補者担当ロンザンか"] = 1.0
rt2 = rt2.loc[:,["案件id（RZ）",'候補者担当ロンザンか']]
rt2["案件id（RZ）"] = rt2["案件id（RZ）"].astype(str)
rt2 = rt2.drop_duplicates(subset=["案件id（RZ）"],keep='first')

rt = pd.merge(rt1,rt2,on = ("案件id（RZ）"),how = "left")
del rt1,rt2

rt["企業フラグ"] = rt["企業担当ロンザンか"] * rt["候補者担当ロンザンか"]
rt["企業フラグ"] = rt["企業フラグ"].apply(lambda x : "両手" if x == 1.0 else "片手")
rt = rt.drop(['企業担当ロンザンか','候補者担当ロンザンか'],axis=1)


In [229]:
#両手企業フラグのデータをポイントデータに紐付け
yomi_data["案件id（RZ）"] = yomi_data["案件id（RZ）"].astype(str)
yomi_data = pd.merge(yomi_data,rt,on = ("案件id（RZ）"),how = "left")

#企業フラグ　→　ヨミ表上の候補者担当と企業担当情報から両手かどうか判断しているフラグ
yomi_data["両手案件フラグ"] = yomi_data.apply(lambda x : "両手" if x["企業フラグ"] == "両手" else "片手" ,axis = 1)

#旧両手　→　本交渉に組手情報がない時の案件、かつ、過去に片手か両手か判断した案件
yomi_data["旧両手"] = yomi_data["旧両手"].fillna(0)
yomi_data["両手案件フラグ"] = yomi_data.apply(lambda x : x["旧両手"] if x["旧両手"] != 0 else x["両手案件フラグ"] ,axis = 1)

#組手　→　本交渉データを元にした組手（組手情報がない場合は、案件データの企業担当＆候補者担当がロンザンメンバーかどうかをみて判断）
yomi_data["組手"] = yomi_data["組手"].fillna(0)
yomi_data["両手案件フラグ"] = yomi_data.apply(lambda x : x["組手"] if x["組手"] != 0 else x["両手案件フラグ"] ,axis = 1)

In [230]:
#yomi_data_ = yomi_data.loc[:,["候補者担当フラグ","企業担当フラグ","氏名","担当所属フラグ","ロンザン所属フラグ",
#                        "候補者アポソース丸め","計上Q","計上月","計上月末","計上週","ポイント丸め","sai_flg","両手案件フラグ",
#                       "ヨミ表選択","期間対象外","Q同営","Q同旬",
#                        "kohosha_id","案件id（RZ）","期","案件\nNo\n（SC）",
#                        "入力者","日付","受注日","クライアント正式名称","候補者","売上種別（商品内容）","基準年収",
#                        "報酬率","新ポイント用","営業売上合計","所属課","氏名","割合","受注額","営業売上\n（営業ポイント）",
#                        "引き継ぎP","担当","備考①","備考②","備考③","内定数フラグ"]]

rzp = yomi_data.rename(columns={"案件\nNo\n（SC）":"案件No（SC）"})

rzp["計上月末"] = pd.to_datetime(rzp["計上月末"])
rzp['内定数フラグ'] = pd.to_numeric(rzp['内定数フラグ'], errors='coerce').fillna(0)
rzp = rzp.astype({'候補者アポソース丸め': str, '計上Q': str, '内定数フラグ': int})



In [231]:
# rzp.to_excel('rzp.xlsx',sheet_name='new_sheet_name')

各数値算出（顧客支持ポイント、成約数）　ロンザンALL用

In [232]:
#APソースが再交渉以外のAPソースの内訳をポイント（エージェントのみ）抽出
#パートナー、人事部、顧問、SMAP、上場企業、転機、その他のポイント（エージェントのみ）抽出
rzp2 = rzp[(rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン") & (~rzp["職種"].str.contains('半常勤', na=False)) & (rzp["売上種別（商品内容）"] != "固定報酬")]
rzp2['type'] = "point_ag"
rzp2.loc[:,"type"] = rzp2["type"].astype(str)
df1 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
# df4 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_all=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_all["type"] = concat_pt_all["type"]+concat_pt_all["候補者アポソース丸め"]

# DataFrameのすべての列をループで確認します。
for col in concat_pt_all.columns:
    # 列のデータ型（dtype）が 'float64' かどうかを判定します。
    if concat_pt_all[col].dtype == 'float64':
        # 'float64' であれば、astype(object) を使って object 型に変換します。
        print(f"列 '{col}' を 'float64' から 'object' に変換します。")
        concat_pt_all[col] = concat_pt_all[col].astype(object)

#APソースが再交渉以外のAPソースの内訳をポイント（エージェントのみ）抽出
#パートナー、人事部、顧問、SMAP、上場企業、転機、その他のポイント（エージェントのみ）抽出
rzp2 = rzp[(rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン") & (~rzp["職種"].str.contains('半常勤', na=False)) & (rzp["売上種別（商品内容）"] == "固定報酬")]
rzp2['type'] = "point_ag_tujo_kotei"
rzp2.loc[:,"type"] = rzp2["type"].astype(str)
df1 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
# df4 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_tujo_kotei=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_tujo_kotei["type"] = concat_pt_tujo_kotei["type"]+concat_pt_tujo_kotei["候補者アポソース丸め"]

#APソースが再交渉以外のAPソースの内訳をポイント（エージェントのみ）抽出
#パートナー、人事部、顧問、SMAP、上場企業、転機、その他のポイント（エージェントのみ）抽出
rzp2 = rzp[(rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン") & (rzp["職種"].str.contains('半常勤', na=False)) & (rzp["売上種別（商品内容）"] != "固定報酬")]
rzp2['type'] = "point_ag_hanjo"
rzp2.loc[:,"type"] = rzp2["type"].astype(str)
df1 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
# df4 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_hanjo=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_hanjo["type"] = concat_pt_hanjo["type"]+concat_pt_hanjo["候補者アポソース丸め"]

#APソースが再交渉以外のAPソースの内訳をポイント（エージェントのみ）抽出
#パートナー、人事部、顧問、SMAP、上場企業、転機、その他のポイント（エージェントのみ）抽出
rzp2 = rzp[(rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン") & (rzp["職種"].str.contains('半常勤', na=False)) & (rzp["売上種別（商品内容）"] == "固定報酬")]
rzp2['type'] = "point_ag_hanjo_kotei"
rzp2.loc[:,"type"] = rzp2["type"].astype(str)
df1 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
# df4 = rzp2.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_hanjo_kotei=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_hanjo_kotei["type"] = concat_pt_hanjo_kotei["type"]+concat_pt_hanjo_kotei["候補者アポソース丸め"]



C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\3751070940.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzp2['type'] = "point_ag"


列 '2016-04-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2016-05-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2016-06-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2016-07-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2016-08-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2016-09-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2016-10-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2016-11-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2016-12-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2017-01-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2017-02-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2017-03-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2017-04-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2017-05-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2017-06-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2017-07-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2017-08-01 00:00:00' を 'float64' から 'object' に変換します。
列 '2017-09-01 00:00:00' を 'float64' から 'object' 

C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\3751070940.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzp2['type'] = "point_ag_tujo_kotei"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\3751070940.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzp2['type'] = "point_ag_hanjo"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\3751070940.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i

In [233]:
# concat_pt_kotei.to_excel('concat_pt_all.xlsx',sheet_name='new_sheet_name')

In [234]:
#APソースが再交渉の内訳をポイント（エージェントのみ）抽出
rzp5 = rzp[(rzp["sai_flg"] == 2) & (rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン") & (~rzp["職種"].str.contains('半常勤', na=False)) & (rzp["売上種別（商品内容）"] != "固定報酬")]
rzp5["type"] = "point_ag_sai"
rzp5.loc[:,"type"] = rzp5["type"].astype(str)
df1 = rzp5.pivot_table(index=["type"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp5.pivot_table(index=["type"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp5.pivot_table(index=["type"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
# df4 = rzp5.pivot_table(index=["type"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_sai=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_sai["type"] = concat_pt_sai["type"]+"再交渉"

#APソースが再交渉の内訳をポイント（エージェントのみ）抽出
rzp5 = rzp[(rzp["sai_flg"] == 2) & (rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン") & (~rzp["職種"].str.contains('半常勤', na=False)) & (rzp["売上種別（商品内容）"] == "固定報酬")]
rzp5["type"] = "point_ag_tujo_kotei_sai"
rzp5.loc[:,"type"] = rzp5["type"].astype(str)
df1 = rzp5.pivot_table(index=["type"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp5.pivot_table(index=["type"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp5.pivot_table(index=["type"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
# df4 = rzp5.pivot_table(index=["type"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_tujo_kotei_sai=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_tujo_kotei_sai["type"] = concat_pt_tujo_kotei_sai["type"]+"再交渉"

#APソースが再交渉の内訳をポイント（エージェントのみ）抽出
rzp5 = rzp[(rzp["sai_flg"] == 2) & (rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン") & (rzp["職種"].str.contains('半常勤', na=False)) & (rzp["売上種別（商品内容）"] != "固定報酬")]
rzp5["type"] = "point_ag_hanjo_sai"
rzp5.loc[:,"type"] = rzp5["type"].astype(str)
df1 = rzp5.pivot_table(index=["type"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp5.pivot_table(index=["type"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp5.pivot_table(index=["type"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
# df4 = rzp5.pivot_table(index=["type"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_hanjo_sai=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_hanjo_sai["type"] = concat_pt_hanjo_sai["type"]+"再交渉"

#APソースが再交渉の内訳をポイント（エージェントのみ）抽出
rzp5 = rzp[(rzp["sai_flg"] == 2) & (rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン") & (rzp["職種"].str.contains('半常勤', na=False)) & (rzp["売上種別（商品内容）"] == "固定報酬")]
rzp5["type"] = "point_ag_hanjo_kotei_sai"
rzp5.loc[:,"type"] = rzp5["type"].astype(str)
df1 = rzp5.pivot_table(index=["type"],columns="計上月末",aggfunc="sum",values="ポイント丸め").fillna(0)
df2 = rzp5.pivot_table(index=["type"],columns="計上Q",aggfunc="sum",values="ポイント丸め").fillna(0)
df3 = rzp5.pivot_table(index=["type"],columns="Q同営",aggfunc="sum",values="ポイント丸め").fillna(0)
# df4 = rzp5.pivot_table(index=["type"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_hanjo_kotei_sai=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_hanjo_kotei_sai["type"] = concat_pt_hanjo_kotei_sai["type"]+"再交渉"

#アポソース別成約数
rzp6 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1)]
rzp6['type'] = "naitei"
rzp6.loc[:,"type"] = rzp6["type"].astype(str)
df1 = rzp6.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzp6.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzp6.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
# df4 = rzp6.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_naitei=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_naitei["type"] = concat_naitei["type"]+concat_naitei["候補者アポソース丸め"]

#片手両手別の成約数
rzp6['type'] = rzp6.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
rzp6.loc[:,"type"] = rzp6["type"].astype(str)
df1 = rzp6.pivot_table(index=["type","両手案件フラグ"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzp6.pivot_table(index=["type","両手案件フラグ"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzp6.pivot_table(index=["type","両手案件フラグ"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
# df4 = rzp6.pivot_table(index=["type","両手案件フラグ"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_kumite=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_kumite["type"] = concat_kumite["type"] + "_seiyaku総計"

#再交渉の成約数
rzp7 = rzp[(rzp["sai_flg"] == 2) & (rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1)]
# rzp7["sai_flg"] = rzp7["sai_flg"].astype(object)
# rzp7.loc[rzp7["sai_flg"] == 2, "sai_flg"] = "再交渉"
rzp7['type'] = "naitei_sai"
rzp7.loc[:,"type"] = rzp7["type"].astype(str)
df1 = rzp7.pivot_table(index=["type","sai_flg"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzp7.pivot_table(index=["type","sai_flg"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzp7.pivot_table(index=["type","sai_flg"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
# df4 = rzp7.pivot_table(index=["type","sai_flg"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_naitei_sai=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_naitei_sai["type"] = concat_naitei_sai["type"]+concat_naitei_sai["sai_flg"].astype(str)

# 1. 'int64'型の列名だけをリストとして抽出
int64_cols = concat_naitei_sai.select_dtypes(include=['int64']).columns

# 2. 抽出した列をまとめて 'object' 型に変換して上書き
concat_naitei_sai[int64_cols] = concat_naitei_sai[int64_cols].astype(object)



C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\3894343605.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzp5["type"] = "point_ag_sai"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\3894343605.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzp5["type"] = "point_ag_tujo_kotei_sai"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\3894343605.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value 

In [235]:
concat_naitei_sai["sai_flg"] = concat_naitei_sai["sai_flg"].astype(str)

In [236]:

#平均基準年収・報酬率抽出
rzp10 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1)]
# rzp10.to_excel('rzp10.xlsx',sheet_name='new_sheet_name')


In [237]:
#基準年収
rzp10['type'] = "income"
rzp10["基準年収"] = pd.to_numeric(rzp10["基準年収"], errors='coerce')
rzp10["基準年収"] = rzp10["基準年収"].fillna(0.0)

#アポソース別平均基準年収抽出（候補者アポソース丸め＝本交渉テーブルのアポソースを、ヨミ表に加工したもの。再交渉案件の場合も、アポソースは元のアポソース(案件に紐づくアポソース)のまま）
rzp10.loc[:,"type"] = rzp10["type"].astype(str)
df1 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
# df4 = rzp10.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_apsource=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_income_apsource["type"] = concat_income_apsource["type"]+concat_income_apsource["候補者アポソース丸め"]

#ロンザン全体の平均基準年収
df1 = rzp10.pivot_table(index=["type"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzp10.pivot_table(index=["type"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzp10.pivot_table(index=["type"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
# df4 = rzp10.pivot_table(index=["type"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_all=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_income_all["type"] = concat_income_all["type"] + "総計"

#片手両手別の平均基準年収
rzp10['type'] = rzp10.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
df1 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
# df4 = rzp10.pivot_table(index=["type","両手案件フラグ"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_kumite=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_income_kumite["type"] = concat_income_kumite["type"] + "_income総計"


C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\1815835719.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzp10['type'] = "income"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\1815835719.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzp10["基準年収"] = pd.to_numeric(rzp10["基準年収"], errors='coerce')
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\1815835719.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_ind

In [238]:
# 重複している列名を削除（最初の列を保持）
rzp = rzp.loc[:, ~rzp.columns.duplicated(keep='first')]

# この後、pd.to_numeric を実行
rzp["報酬率"] = pd.to_numeric(rzp["報酬率"], errors='coerce')

#報酬率
rzp11 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1) & (rzp["報酬率"] < 1)]

#報酬率
rzp11['type'] = "hoshu_ritsu"
rzp11["報酬率"] = rzp11["報酬率"].fillna(0.0)

#アポソース別報酬率抽出（候補者アポソース丸め＝本交渉テーブルのアポソースを、ヨミ表に加工したもの。再交渉案件の場合も、アポソースは元のアポソース(案件に紐づくアポソース)のまま）
rzp11.loc[:,"type"] = rzp11["type"].astype(str)
df1 = rzp11.pivot_table(index=["type","候補者アポソース丸め"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzp11.pivot_table(index=["type","候補者アポソース丸め"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzp11.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
# df4 = rzp11.pivot_table(index=["type","候補者アポソース丸め"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_apsource=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_hoshu_ritsu_apsource["type"] = concat_hoshu_ritsu_apsource["type"]+concat_hoshu_ritsu_apsource["候補者アポソース丸め"]

#ロンザン全体の報酬率
df1 = rzp11.pivot_table(index=["type"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzp11.pivot_table(index=["type"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzp11.pivot_table(index=["type"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
# df4 = rzp11.pivot_table(index=["type"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_all=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_hoshu_ritsu_all["type"] = concat_hoshu_ritsu_all["type"] + "総計"

#片手両手別の報酬率
rzp11['type'] = rzp11.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
df1 = rzp11.pivot_table(index=["type","両手案件フラグ"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzp11.pivot_table(index=["type","両手案件フラグ"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzp11.pivot_table(index=["type","両手案件フラグ"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
# df4 = rzp11.pivot_table(index=["type","両手案件フラグ"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_kumite=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_hoshu_ritsu_kumite["type"] = concat_hoshu_ritsu_kumite["type"] + "_hoshu_ritsu総計"

C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\2559282163.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzp11['type'] = "hoshu_ritsu"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\2559282163.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzp11["報酬率"] = rzp11["報酬率"].fillna(0.0)
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\2559282163.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value 

個人別数値算出用

In [239]:
rzp = rzp.rename(columns={"氏名":"sei_plus"})

In [240]:
#個人別顧客支持pt（クロスセルも含む※ただし、BT・RPAは除く）
rzpk2 = rzp[(rzp["期間対象外"] == "対象") & (rzp["所属課"] == "ロンザン") & (rzp["ロンザン所属フラグ"] == "1")]
rzpk2['type'] = "point_ag"
rzpk2.loc[:,"type"] = rzpk2["type"].astype(str)
df1 = rzpk2.pivot_table(index=["type","sei_plus"],columns="計上月末",aggfunc="sum",values="ポイント丸め(組織貢献引当後)").fillna(0)
df2 = rzpk2.pivot_table(index=["type","sei_plus"],columns="計上Q",aggfunc="sum",values="ポイント丸め(組織貢献引当後)").fillna(0)
df3 = rzpk2.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="sum",values="ポイント丸め(組織貢献引当後)").fillna(0)
# df4 = rzpk2.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_kojin["type"] = concat_pt_kojin["type"]+concat_pt_kojin["sei_plus"].astype(str)

C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\3408861670.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzpk2['type'] = "point_ag"


In [241]:
rzpk2point_ag_df = rzpk2[rzpk2['計上Q'].str.contains('29-1Q', na=False)]

In [242]:


#クロスセルポイント抽出
rzpk4 = rzp[ (rzp["期間対象外"] == "対象") & (rzp["候補者アポソース丸め"] == "■クロスセル")  & (rzp["所属課"] == "ロンザン") & (rzp["ロンザン所属フラグ"] == "1")]
rzpk4['type'] = "point_cross_all"
rzpk4.loc[:,"type"] = rzpk4["type"].astype(str)
df1 = rzpk4.pivot_table(index=["type","sei_plus"],columns="計上月末",aggfunc="sum",values="ポイント丸め(組織貢献引当後)").fillna(0)
df2 = rzpk4.pivot_table(index=["type","sei_plus"],columns="計上Q",aggfunc="sum",values="ポイント丸め(組織貢献引当後)").fillna(0)
df3 = rzpk4.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="sum",values="ポイント丸め(組織貢献引当後)").fillna(0)
# df4 = rzpk4.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="sum",values="ポイント丸め").fillna(0)
concat_pt_cross_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_pt_cross_kojin["type"] = concat_pt_cross_kojin["type"]+concat_pt_cross_kojin["sei_plus"].astype(str)

#候補者担当別成約数
rzpk6 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1) & (rzp["ロンザン所属フラグ"] == "1")]

rzpk6['type'] = "naitei"
rzpk6.loc[:,"type"] = rzpk6["type"].astype(str)
df1 = rzpk6.pivot_table(index=["type","sei_plus"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzpk6.pivot_table(index=["type","sei_plus"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzpk6.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
# df4 = rzpk6.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_naitei_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_naitei_kojin["type"] = concat_naitei_kojin["type"]+concat_naitei_kojin["sei_plus"].astype(str)

#片手両手別の成約数
rzpk6['type'] = rzpk6.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
rzpk6.loc[:,"type"] = rzpk6["type"].astype(str)
df1 = rzpk6.pivot_table(index=["type","両手案件フラグ","sei_plus"],columns="計上月末",aggfunc="count",values="内定数フラグ").fillna(0)
df2 = rzpk6.pivot_table(index=["type","両手案件フラグ","sei_plus"],columns="計上Q",aggfunc="count",values="内定数フラグ").fillna(0)
df3 = rzpk6.pivot_table(index=["type","両手案件フラグ","sei_plus"],columns="Q同営",aggfunc="count",values="内定数フラグ").fillna(0)
# df4 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="Q同旬",aggfunc="count",values="内定数フラグ").fillna(0)
concat_kumite_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_kumite_kojin["type"] = concat_kumite_kojin["type"] + "_seiyaku" + concat_kumite_kojin["sei_plus"].astype(str)

#候補者担当別平均基準年収
rzpk6['type'] = "income"
rzpk6["基準年収"] = rzpk6["基準年収"].astype(str).str.strip()
rzpk6["基準年収"] = pd.to_numeric(rzpk6["基準年収"], errors='coerce')
df1 = rzpk6.pivot_table(index=["type","sei_plus"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzpk6.pivot_table(index=["type","sei_plus"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzpk6.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
# df4 = rzpk6.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_income_kojin["type"] = concat_income_kojin["type"] + concat_income_kojin["sei_plus"].astype(str)

#片手両手別の報酬率
rzpk7 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1) & (rzp["報酬率"] < 1) & (rzp["ロンザン所属フラグ"] == "1")]
rzpk7['type'] = "hoshu_ritsu"
df1 = rzpk7.pivot_table(index=["type","sei_plus"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzpk7.pivot_table(index=["type","sei_plus"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzpk7.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
# df4 = rzpk7.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_hoshu_ritsu_kojin["type"] = concat_hoshu_ritsu_kojin["type"] + concat_hoshu_ritsu_kojin["sei_plus"].astype(str)


#片手両手別の平均基準年収
rzpk6['type'] = rzpk6.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
df1 = rzpk6.pivot_table(index=["type","両手案件フラグ","sei_plus"],columns="計上月末",aggfunc="mean",values="基準年収").fillna(0)
df2 = rzpk6.pivot_table(index=["type","両手案件フラグ","sei_plus"],columns="計上Q",aggfunc="mean",values="基準年収").fillna(0)
df3 = rzpk6.pivot_table(index=["type","両手案件フラグ","sei_plus"],columns="Q同営",aggfunc="mean",values="基準年収").fillna(0)
# df4 = rzpk6.pivot_table(index=["type","両手案件フラグ","氏名"],columns="Q同旬",aggfunc="mean",values="基準年収").fillna(0)
concat_income_kumite_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_income_kumite_kojin["type"] = concat_income_kumite_kojin["type"].astype(str) + "_income" + concat_income_kumite_kojin["sei_plus"].astype(str)

#片手両手別の報酬率
rzpk7['type'] = rzpk7.apply(lambda x : "ryoute" if x["両手案件フラグ"] == "両手" else "katate" ,axis = 1)
df1 = rzpk7.pivot_table(index=["type","両手案件フラグ","sei_plus"],columns="計上月末",aggfunc="mean",values="報酬率").fillna(0)
df2 = rzpk7.pivot_table(index=["type","両手案件フラグ","sei_plus"],columns="計上Q",aggfunc="mean",values="報酬率").fillna(0)
df3 = rzpk7.pivot_table(index=["type","両手案件フラグ","sei_plus"],columns="Q同営",aggfunc="mean",values="報酬率").fillna(0)
# df4 = rzpk7.pivot_table(index=["type","両手案件フラグ","氏名"],columns="Q同旬",aggfunc="mean",values="報酬率").fillna(0)
concat_hoshu_ritsu_kumite_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_hoshu_ritsu_kumite_kojin["type"] = concat_hoshu_ritsu_kumite_kojin["type"].astype(str) + "_hoshu_ritsu" + concat_hoshu_ritsu_kumite_kojin["sei_plus"].astype(str)



C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\443892225.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzpk4['type'] = "point_cross_all"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\443892225.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzpk6['type'] = "naitei"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\443892225.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

In [243]:
#企業担当別成約数
rzpk8 = rzp[(rzp["期間対象外"] == "対象") & (rzp["企業担当フラグ"] == 1) & (rzp["ロンザン所属フラグ"] == "1")]
rzpk8['type'] = "naitei_kigyo"
rzpk8.loc[:,"type"] = rzpk8["type"].astype(str)
rzpk8 = rzpk8[["type","案件id（RZ）","案件No（SC）","sei_plus","計上月末","計上Q","Q同営","企業担当フラグ"]]
rzpk8 = rzpk8.drop_duplicates()

C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\4284870389.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzpk8['type'] = "naitei_kigyo"


In [244]:

df1 = rzpk8.pivot_table(index=["type","sei_plus"],columns="計上月末",aggfunc="count",values="企業担当フラグ").fillna(0)
df2 = rzpk8.pivot_table(index=["type","sei_plus"],columns="計上Q",aggfunc="count",values="企業担当フラグ").fillna(0)
df3 = rzpk8.pivot_table(index=["type","sei_plus"],columns="Q同営",aggfunc="count",values="企業担当フラグ").fillna(0)
# df4 = rzpk7.pivot_table(index=["type","氏名"],columns="Q同旬",aggfunc="count",values="企業担当フラグ").fillna(0)
concat_naitei_kigyo_kojin=  pd.concat([df1,df2,df3],axis=1).reset_index()
concat_naitei_kigyo_kojin["type"] = concat_naitei_kigyo_kojin["type"]+concat_naitei_kigyo_kojin["sei_plus"].astype(str)


In [245]:

# rzpk8.to_excel('rzpk8.xlsx',sheet_name='new_sheet_name')


カレンダー用

In [246]:
#候補者担当×アポソース別成約数（再交渉になった案件は、元のアポソース側ではなく、再交渉側でカウントするため、再交渉案件を除く）
rzpc5 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1) & (rzp["sai_flg"] != 1) & (rzp["ロンザン所属フラグ"] == "1")]
rzpc5['type'] = "naitei"
rzpc5.loc[:,"type"] = rzpc5["type"].astype(str)
calendar_naitei_cal_kojin = rzpc5.pivot_table(index=["type","sei_plus","候補者アポソース丸め"],columns="計上週",aggfunc="count",values="内定数フラグ").fillna(0).reset_index()
calendar_naitei_cal_kojin["type"] = calendar_naitei_cal_kojin["type"]+calendar_naitei_cal_kojin["候補者アポソース丸め"]+calendar_naitei_cal_kojin["sei_plus"].astype(str)

#候補者担当×再交渉成約数
rzpc6 = rzp[(rzp["期間対象外"] == "対象") & (rzp["内定数フラグ"] == 1) & (rzp["sai_flg"] == 2) & (rzp["ロンザン所属フラグ"] == "1")]
rzpc6['type'] = "naitei再交渉"
rzpc6.loc[:,"type"] = rzpc6["type"].astype(str)
calendar_naitei_cal_sai = rzpc6.pivot_table(index=["type","sei_plus"],columns="計上週",aggfunc="count",values="内定数フラグ").fillna(0).reset_index()
calendar_naitei_cal_sai["type"] = calendar_naitei_cal_sai["type"]+calendar_naitei_cal_sai["sei_plus"].astype(str)


#候補者担当×再交渉成約数
rzpc7 = rzp[(rzp["期間対象外"] == "対象") & (rzp["企業担当フラグ"] == 1) & (rzp["ロンザン所属フラグ"] == "1")]
rzpc7['type'] = "naitei_kigyo"
rzpc7.loc[:,"type"] = rzpc7["type"].astype(str)
calendar_naitei_cal_kigyo = rzpc7.pivot_table(index=["type","sei_plus"],columns="計上週",aggfunc="count",values="企業担当フラグ").fillna(0).reset_index()
calendar_naitei_cal_kigyo["type"] = calendar_naitei_cal_kigyo["type"]+calendar_naitei_cal_kigyo["sei_plus"].astype(str)


C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\4047254134.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzpc5['type'] = "naitei"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\4047254134.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rzpc6['type'] = "naitei再交渉"
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\4047254134.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the c

In [247]:
# 結合予定のデータフレームをリストにまとめる
dfs_to_convert = [
    concat_pt_all,
    concat_pt_tujo_kotei,
    concat_pt_hanjo,
    concat_pt_hanjo_kotei,
    concat_pt_sai,
    concat_pt_tujo_kotei_sai,
    concat_pt_hanjo_sai,
    concat_pt_hanjo_kotei_sai,
    concat_naitei,
    concat_naitei_sai,
    concat_kumite,
    concat_income_apsource,
    concat_income_all,
    concat_income_kumite,
    concat_hoshu_ritsu_apsource,
    concat_hoshu_ritsu_all,
    concat_hoshu_ritsu_kumite
]

# リスト内のすべてのデータフレームに対して、float64 の列を object に一括変換
for df in dfs_to_convert:
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype(object)

In [248]:
concat_naitei.dtypes


type                   object
候補者アポソース丸め             object
2016-04-01 00:00:00    object
2016-05-01 00:00:00    object
2016-06-01 00:00:00    object
2016-07-01 00:00:00    object
2016-08-01 00:00:00    object
2016-09-01 00:00:00    object
2016-10-01 00:00:00    object
2016-11-01 00:00:00    object
2016-12-01 00:00:00    object
2017-01-01 00:00:00    object
2017-02-01 00:00:00    object
2017-03-01 00:00:00    object
2017-04-01 00:00:00    object
2017-05-01 00:00:00    object
2017-06-01 00:00:00    object
2017-07-01 00:00:00    object
2017-08-01 00:00:00    object
2017-09-01 00:00:00    object
2017-10-01 00:00:00    object
2017-11-01 00:00:00    object
2017-12-01 00:00:00    object
2018-01-01 00:00:00    object
2018-02-01 00:00:00    object
2018-03-01 00:00:00    object
2018-04-01 00:00:00    object
2018-05-01 00:00:00    object
2018-06-01 00:00:00    object
2018-07-01 00:00:00    object
2018-08-01 00:00:00    object
2018-09-01 00:00:00    object
2018-10-01 00:00:00    object
2018-11-01

In [249]:
concat_naitei_sai.dtypes


type                   object
sai_flg                object
2017-09-01 00:00:00    object
2018-11-01 00:00:00    object
2019-03-01 00:00:00    object
2019-06-01 00:00:00    object
2019-08-01 00:00:00    object
2019-09-01 00:00:00    object
2019-10-01 00:00:00    object
2019-11-01 00:00:00    object
2019-12-01 00:00:00    object
2020-01-01 00:00:00    object
2020-02-01 00:00:00    object
2020-03-01 00:00:00    object
2020-04-01 00:00:00    object
2020-05-01 00:00:00    object
2020-07-01 00:00:00    object
2020-08-01 00:00:00    object
2020-09-01 00:00:00    object
2020-10-01 00:00:00    object
2020-11-01 00:00:00    object
2020-12-01 00:00:00    object
2021-01-01 00:00:00    object
2021-02-01 00:00:00    object
2021-03-01 00:00:00    object
2021-04-01 00:00:00    object
2021-05-01 00:00:00    object
2021-06-01 00:00:00    object
2021-08-01 00:00:00    object
2021-09-01 00:00:00    object
2021-10-01 00:00:00    object
2021-11-01 00:00:00    object
2021-12-01 00:00:00    object
2022-01-01

In [250]:
#ロンザンALL用
concat_all = pd.merge(concat_all,concat_pt_all,how = "outer")
concat_all = pd.merge(concat_all,concat_pt_tujo_kotei,how = "outer")
concat_all = pd.merge(concat_all,concat_pt_hanjo,how = "outer")
concat_all = pd.merge(concat_all,concat_pt_hanjo_kotei,how = "outer")

concat_all = pd.merge(concat_all,concat_pt_sai,how = "outer")
concat_all = pd.merge(concat_all,concat_pt_tujo_kotei_sai,how = "outer")
concat_all = pd.merge(concat_all,concat_pt_hanjo_sai,how = "outer")
concat_all = pd.merge(concat_all,concat_pt_hanjo_kotei_sai,how = "outer")

concat_all = pd.merge(concat_all,concat_naitei,how = "outer")
concat_all = pd.merge(concat_all,concat_naitei_sai,how = "outer")
concat_all = pd.merge(concat_all,concat_kumite,how = "outer")
concat_all = pd.merge(concat_all,concat_income_apsource,how = "outer")
concat_all = pd.merge(concat_all,concat_income_all,how = "outer")
concat_all = pd.merge(concat_all,concat_income_kumite,how = "outer")
concat_all = pd.merge(concat_all,concat_hoshu_ritsu_apsource,how = "outer")
concat_all = pd.merge(concat_all,concat_hoshu_ritsu_all,how = "outer")
concat_all = pd.merge(concat_all,concat_hoshu_ritsu_kumite,how = "outer")
filtered_data = concat_all[concat_all['type'].str.contains('point_ag', na=False)]
filtered_data

,type,APソース丸め,2017-04-01 00:00:00,2017-05-01 00:00:00,2017-06-01 00:00:00,2017-07-01 00:00:00,2017-08-01 00:00:00,2017-09-01 00:00:00,2017-10-01 00:00:00,2017-11-01 00:00:00,2017-12-01 00:00:00,2018-01-01 00:00:00,2018-02-01 00:00:00,2018-03-01 00:00:00,2018-04-01 00:00:00,2018-05-01 00:00:00,2018-06-01 00:00:00,2018-07-01 00:00:00,2018-08-01 00:00:00,2018-09-01 00:00:00,2018-10-01 00:00:00,2018-11-01 00:00:00,2018-12-01 00:00:00,2019-01-01 00:00:00,2019-02-01 00:00:00,2019-03-01 00:00:00,2019-04-01 00:00:00,2019-05-01 00:00:00,2019-06-01 00:00:00,2019-07-01 00:00:00,2019-08-01 00:00:00,2019-09-01 00:00:00,2019-10-01 00:00:00,2019-11-01 00:00:00,2019-12-01 00:00:00,2020-01-01 00:00:00,2020-02-01 00:00:00,2020-03-01 00:00:00,2020-04-01 00:00:00,2020-05-01 00:00:00,2020-06-01 00:00:00,2020-07-01 00:00:00,2020-08-01 00:00:00,2020-09-01 00:00:00,2020-10-01 00:00:00,2020-11-01 00:00:00,2020-12-01 00:00:00,2021-01-01 00:00:00,2021-02-01 00:00:00,2021-03-01 00:00:00,2021-04-01 00:00:00,2021-05-01 00:00:00,2021-06-01 00:00:00,2021-07-01 00:00:00,2021-08-01 00:00:00,2021-09-01 00:00:00,2021-10-01 00:00:00,2021-11-01 00:00:00,2021-12-01 00:00:00,2022-01-01 00:00:00,2022-02-01 00:00:00,2022-03-01 00:00:00,2022-04-01 00:00:00,2022-05-01 00:00:00,2022-06-01 00:00:00,2022-07-01 00:00:00,2022-08-01 00:00:00,2022-09-01 00:00:00,2022-10-01 00:00:00,2022-11-01 00:00:00,2022-12-01 00:00:00,2023-01-01 00:00:00,2023-02-01 00:00:00,2023-03-01 00:00:00,2023-04-01 00:00:00,2023-05-01 00:00:00,2023-06-01 00:00:00,2023-07-01 00:00:00,2023-08-01 00:00:00,2023-09-01 00:00:00,2023-10-01 00:00:00,2023-11-01 00:00:00,2023-12-01 00:00:00,2024-01-01 00:00:00,2024-02-01 00:00:00,2024-03-01 00:00:00,2024-04-01 00:00:00,2024-05-01 00:00:00,2024-06-01 00:00:00,2024-07-01 00:00:00,2024-08-01 00:00:00,2024-09-01 00:00:00,2024-10-01 00:00:00,2024-11-01 00:00:00,2024-12-01 00:00:00,2025-01-01 00:00:00,2025-02-01 00:00:00,2025-03-01 00:00:00,2025-04-01 00:00:00,2025-05-01 00:00:00,...,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q,29-4Q,20-3Q同営業日,20-4Q同営業日,21-1Q同営業日,21-2Q同営業日,21-3Q同営業日,21-4Q同営業日,22-1Q同営業日,22-2Q同営業日,22-3Q同営業日,22-4Q同営業日,23-1Q同営業日,23-2Q同営業日,23-3Q同営業日,23-4Q同営業日,24-1Q同営業日,24-2Q同営業日,24-3Q同営業日,24-4Q同営業日,25-1Q同営業日,25-2Q同営業日,25-3Q同営業日,25-4Q同営業日,26-1Q同営業日,26-2Q同営業日,26-3Q同営業日,26-4Q同営業日,27-1Q同営業日,27-2Q同営業日,27-3Q同営業日,27-4Q同営業日,28-1Q同営業日,28-2Q同営業日,28-3Q同営業日,28-4Q同営業日,29-1Q同営業日,29-2Q同営業日,29-3Q同営業日,29-4Q同営業日,組手,候補者アポソース丸め,2016-04-01 00:00:00,2016-05-01 00:00:00,2016-06-01 00:00:00,2016-07-01 00:00:00,2016-08-01 00:00:00,2016-09-01 00:00:00,2016-10-01 00:00:00,2016-11-01 00:00:00,2016-12-01 00:00:00,2017-01-01 00:00:00,2017-02-01 00:00:00,2017-03-01 00:00:00,19-3Q,19-4Q,20-1Q,20-2Q,19-3Q同営業日,19-4Q同営業日,20-1Q同営業日,20-2Q同営業日,sai_flg,両手案件フラグ
144,point_agSMAP,NaN,167.479301,75.486757,184.771828,259.4592,645.832858,680.112385,0.0,210.09885,335.904975,0.0,0.0,457.683223,257.094489,0.0,499.56368,0.0,156.600444,742.565779,0.0,303.325044,457.914573,211.422996,280.58919,583.815896,177.511379,688.824565,1378.282648,782.86371,922.414301,806.169015,372.85472,316.321626,355.49304,201.813242,1067.390218,1838.728964,277.633138,102.55896,1443.369305,487.476101,1284.070639,1168.391619,487.963653,233.7741,1474.223287,1726.984948,199.556835,2974.024598,554.20577,1108.420227,2111.603802,568.060248,377.912782,1139.831263,561.405452,1118.673638,1795.874962,329.764574,581.431738,1650.014849,598.456136,413.106843,1024.353983,956.336792,172.368,2730.415964,1113.131156,863.244736,2918.227697,1525.610831,1937.902033,3263.067984,2488.822702,1293.065141,3086.026143,3899.984835,3274.943517,6758.447525,2156.865,1938.943,5639.511,1653.669,3135.768,5615.651,366.693,3397.627,6334.945,6608.044,3841.903,9416.626,1806.081,2949.713,3193.889,4493.705,2852.051,5550.448,3593.042,4108.501,...,427.737887,1585.404443,546.003825,457.683223,756

In [251]:
#個人別用
concat_kojin = pd.merge(concat_kojin,concat_pt_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_pt_cross_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_naitei_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_naitei_kigyo_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_kumite_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_income_kumite_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_hoshu_ritsu_kumite_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_income_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_hoshu_ritsu_kojin,how = "outer")
concat_kojin = pd.merge(concat_kojin,concat_hoshu_ritsu_kojin,how = "outer")



In [252]:
mas2 = master2[["sei_plus","user_id"]].drop_duplicates()

In [253]:
filtered_concat_kojin = pd.merge(concat_kojin,mas2,on = "sei_plus",how="left")


In [254]:
filtered_concat_kojin = filtered_concat_kojin.dropna(subset=['user_id'])

# filtered_concat_kojin = filtered_concat_kojin[(filtered_concat_kojin["user_id"] == "1")] 

In [255]:
point_ag_df = filtered_concat_kojin[filtered_concat_kojin['type'].str.contains('point_ag', na=False)]

# 抽出結果の確認（任意）

In [256]:
import re

# 正規表現パターンは同じです
pattern = r'^\d{2}-\dQ$'

# カラムを抽出する条件を変更します
columns_to_keep = [
    col for col in filtered_concat_kojin.columns 
    if col in ['type', 'sei_plus'] or re.match(pattern, str(col))
]

# --- 修正部分はここまで ---


# 抽出したカラム名のリストでDataFrameをフィルタリングします
final_df = filtered_concat_kojin[columns_to_keep]

In [257]:
# mas2
# master2
# concat_kojin.sample(10)
# filtered_concat_kojin.sample(10)
final_df.sample(10)

,type,sei_plus,20-3Q,20-4Q,21-1Q,21-2Q,21-3Q,21-4Q,22-1Q,22-2Q,22-3Q,22-4Q,23-1Q,23-2Q,23-3Q,23-4Q,24-1Q,24-2Q,24-3Q,24-4Q,25-1Q,25-2Q,25-3Q,25-4Q,26-1Q,26-2Q,26-3Q,26-4Q,27-1Q,27-2Q,27-3Q,27-4Q,28-1Q,28-2Q,28-3Q,28-4Q,29-1Q,29-2Q,29-3Q,29-4Q,19-3Q,19-4Q,20-1Q,20-2Q
15766,naitei阿曽祐,阿曽祐,5.0,5.0,3.0,2.0,2.0,9.0,5.0,5.0,3.0,5.0,4.0,6.0,2.0,3.0,3.0,2.0,2.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,4.0,3.0
968,eigyou仙頭克,仙頭克,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
4965,jissi両手中島健,中島健,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
21855,shoki_ap中野理,中野理,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,1.0,1.0,0.0,1.0,0.0,1.0,49.0,4.0,31.0,113.0,10.0,5.0,19.0,2.0,11.0,34.0,14.0,42.0,9.0,75.0,3.0,0.0,0.0,NaN,NaN,NaN,NaN
20947,settei片手松田和,松田和,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,16.0,9.0,6.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
3751,jissi_sai片手山内瑛,山内瑛,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,0.0,4.0,2.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
17200,settei_com片手川田剛,川田剛,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
21891,shoki_ap内田の,内田の,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,4.0,5.0,1.0,1.0,1.0,2.0,1.0,0.0,30.0,69.0,23.0,NaN,NaN,NaN,NaN
3367,jissi_sai両手花田和,花田和,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,4.0,5.0,3.0,9.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
5495,jissi片手坂巻汐,坂巻汐,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,5.0,8.0,6.0,5.0,10.0,8.0,7.0,6.0,2.0,7.0,4.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [258]:
#カレンダー用
calendar = pd.merge(calendar,calendar_naitei_cal_kojin,how = "outer")
calendar = pd.merge(calendar,calendar_naitei_cal_sai,how = "outer")
calendar = pd.merge(calendar,calendar_naitei_cal_kigyo,how = "outer")

In [259]:
filtered_mas2 = master2[(master2['ロンザン所属フラグ'] == "1")& (master2['Q'] == Q)]
joined_calendar = pd.merge(calendar,filtered_mas2,on = "sei_plus",how="left")
filtered_calendar = joined_calendar[(joined_calendar['ロンザン所属フラグ']=="1")]

In [260]:
columns_to_drop = [
    'APソース丸め', '候補者アポソース丸め', 'Q', '人マスタ', 'user_id', 
    '所属フラグ', 'ロンザン所属フラグ', 'レイヤー'
]

In [261]:
final_calendar = filtered_calendar.drop(columns=columns_to_drop, errors='ignore')

In [262]:
with pd.ExcelWriter('index_NEW.xlsx', engine = 'xlsxwriter') as writer:
    concat_all.to_excel(writer,sheet_name='all')
    filtered_concat_kojin.to_excel(writer,sheet_name='kojin')
    final_calendar.to_excel(writer,sheet_name='calendar')

In [263]:
import python_ss.python_ss as ps


In [264]:
# 欠損値などを置き換える処理
concat_all.replace([np.inf, -np.inf], np.nan, inplace=True)
concat_all = concat_all.astype(str)
concat_all.replace('nan', '', inplace=True)

# --- ここからが修正部分 ---

# 1. 【重要】カラム名を一つずつ文字列に変換しながらヘッダーリストを作成
header = [str(col) for col in concat_all.columns]

# --- 修正部分はここまで ---

# 2. データをリストのリストとして取得
data = concat_all.values.tolist()

# 3. ヘッダーとデータを結合
editted_concat_all = [header] + data

# スプレッドシートに記載する
SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
json_path = r"Z:\Users\suehara\Documents\python\analysis\python_ss\credentials.json"
service = ps.get_auth(SCOPES, json_path)
SPREADSHEET_ID = '1Pe_gU9i3k97_KOXfzeN-PV74RGCixQJCw3DmO_9H7Sw'
Sheet_NAME = 'index_all!A'
Sheet_row = "1"
RANGE_NAME = Sheet_NAME + Sheet_row
ps.update_ss(SPREADSHEET_ID, RANGE_NAME, editted_concat_all, service)

C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\3437826952.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  concat_all.replace([np.inf, -np.inf], np.nan, inplace=True)


In [265]:
# 欠損値などを置き換える処理（この部分は変更なし）
final_df.replace([np.inf, -np.inf], np.nan, inplace=True)
final_df.fillna('', inplace=True)

# --- ここからが修正部分 ---

# 1. ヘッダーをリストとして取得
header = final_df.columns.tolist()

# 2. データをリストのリストとして取得
data = final_df.values.tolist()

# 3. ヘッダーとデータを結合して、書き込むための完全なリストを作成
values_to_write = [header] + data

# --- 修正部分はここまで ---


# スプレッドシートの情報を設定（この部分は変更なし）
SPREADSHEET_ID = '1Fq7rbtt6--bkl-laypFFPqYDuTvy_Vps4p30j7FoSS4'
Sheet_NAME = 'kojin!F'
Sheet_row = "1"
RANGE_NAME = Sheet_NAME + Sheet_row

# ps.update_ssに関数に、作成した `values_to_write` を渡します
# ※渡す変数を `filtered_concat_kojin` から `values_to_write` に変更しています
ps.update_ss(SPREADSHEET_ID, RANGE_NAME, values_to_write, service)

C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\4287889706.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\4287889706.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  final_df.fillna('', inplace=True)
C:\Users\suehara\AppData\Local\Temp\ipykernel_15020\4287889706.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.fill

In [268]:
# 欠損値などを置き換える処理
filtered_concat_kojin.replace([np.inf, -np.inf], np.nan, inplace=True)
filtered_concat_kojin = filtered_concat_kojin.astype(str)
filtered_concat_kojin.replace('nan', '', inplace=True)

# --- ここからが修正部分 ---

# 1. 【重要】カラム名を一つずつ文字列に変換しながらヘッダーリストを作成
header = [str(col) for col in filtered_concat_kojin.columns]

# --- 修正部分はここまで ---

# 2. データをリストのリストとして取得
data = filtered_concat_kojin.values.tolist()

# 3. ヘッダーとデータを結合
editted_filtered_concat_kojin = [header] + data

# スプレッドシートに記載する
Sheet_NAME = 'index_kojin!A'
Sheet_row = "1"
RANGE_NAME = Sheet_NAME+Sheet_row
ps.update_ss(SPREADSHEET_ID,RANGE_NAME,editted_filtered_concat_kojin,service)



SSLEOFError: EOF occurred in violation of protocol (_ssl.c:2427)

In [ ]:
# 欠損値などを置き換える処理（この部分は変更なし）
final_calendar.replace([np.inf, -np.inf], np.nan, inplace=True)
final_calendar.fillna('', inplace=True)

# --- ここからが修正部分 ---

# 1. ヘッダーをリストとして取得
header = final_calendar.columns.tolist()

# 2. データをリストのリストとして取得
data = final_calendar.values.tolist()

# 3. ヘッダーとデータを結合して、書き込むための完全なリストを作成
values_to_write = [header] + data

# --- 修正部分はここまで ---


# スプレッドシートの情報を設定（この部分は変更なし）
SPREADSHEET_ID = '1edOvnfVhSsBYZXHMoNUMsUPLMZLtJGhYrRLHZREqPfE'
Sheet_NAME = 'ピボット!A'
Sheet_row = "2"
RANGE_NAME = Sheet_NAME + Sheet_row

# ps.update_ssに関数に、作成した `values_to_write` を渡します
# ※渡す変数を `filtered_concat_kojin` から `values_to_write` に変更しています
ps.update_ss(SPREADSHEET_ID, RANGE_NAME, values_to_write, service)